In [1]:
from typing import TypedDict ,List ,Union
from langchain_core.messages import HumanMessage ,AIMessage
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph ,START ,END
from dotenv import load_dotenv
from typing import Annotated , Sequence ,TypedDict
from langchain_core.messages import BaseMessage ,ToolMessage ,SystemMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph  ,END ,START
from langgraph.prebuilt import ToolNode
from langchain.chat_models import init_chat_model

In [2]:
# !pip install langchain_chroma

In [3]:
import os
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
llm = init_chat_model(
    model="llama3.1:latest ",#phi3.5",  # The name of the model in Ollama #'deepseek-r1:1.5b',#
    model_provider="ollama",  # Explicitly state the provider
    temperature=0.7,  # Optional: control creativity
    # base_url="http://localhost:11434" # Optional: If your Ollama is on a different host/port
)

In [5]:
PDF_PATH = "eg_asp.pdf"#"IPA1.pdf"
LLM_MODEL = "llama3.1:latest"
EMBEDDING_MODEL = "mxbai-embed-large"

In [6]:
def setup_rag_chain(pdf_path: str):
    # Load the PDF document
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    docs = text_splitter.split_documents(documents)
    
    if not docs:
        print("Error: The PDF file is empty or could not be processed.")
        return None

    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

    # Store the chunks and embeddings in ChromaDB
    print("Step 4: Creating a ChromaDB vector store...")
    vector_store = Chroma.from_documents(documents=docs, embedding=embeddings)
    
    # Create a retriever
    retriever = vector_store.as_retriever(search_kwargs={"k": 5})

    llm = Ollama(model=LLM_MODEL)

    # Create the prompt template
    prompt = ChatPromptTemplate.from_template("""
    You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know. 
    
    Question: {question}
    Context: {context}
    Answer:
    """)


    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

In [7]:
def chat_with_pdf():
    """
    Main function to run the PDF chatbot.
    """
    rag_chain = setup_rag_chain(PDF_PATH)
    
    if rag_chain is None:
        return

    print("\n-------------------------------------------")
    print(f"Chatbot is ready. You can now ask questions about '{os.path.basename(PDF_PATH)}'.")
    print("Type 'exit' to quit.")
    print("-------------------------------------------")
    
    while True:
        user_question = input("\nYour question: ")
        if user_question.lower() == 'exit':
            print("Goodbye!")
            break
        
        print("Thinking...")
        try:
            response = rag_chain.invoke(user_question)
            print(f"\nChatbot: {response}")
        except Exception as e:
            print(f"An error occurred: {e}")
            print("Please make sure the Ollama server is running and the models are pulled.")
            break

In [8]:
# !pip install pypdf

In [9]:
if __name__ == "__main__":
    # Before running this script:
    # 1. Make sure you have Ollama installed and running.
    # 2. Pull the required models by running 'ollama pull llama3' and 'ollama pull nomic-embed-text'
    #    in your terminal.
    # 3. Place your PDF file in the same directory as this script and rename it to 'your_document.pdf'.
    
    # Start the chatbot
    chat_with_pdf()


ImportError: `pypdf` package not found, please install it with `pip install pypdf`

In [ ]:
import operator
from typing import TypedDict, Sequence, Annotated

# Import necessary components from langchain and langgraph
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_community.chat_models import ChatOllama
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode

# 1. Define the AgentState
# The 'messages' key will accumulate BaseMessage objects.
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    # We'll add a 'context' field to demonstrate injecting arbitrary data
    context: Annotated[str | None, operator.setitem] # Use setitem to replace context

# 2. Define a simple tool
@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two integers together."""
    return a * b

tools = [multiply]

# 3. Initialize the Language Model
# Ensure Ollama is running and 'llama3.2' model is available
try:
    llm = ChatOllama(model='llama3.2', temperature=0.7).bind_tools(tools)
    model = llm
except Exception as e:
    print(f"Could not connect to Ollama or model not found: {e}")
    print("Falling back to a dummy model for demonstration purposes.")
    # Fallback for demonstration if Ollama isn't set up
    from langchain_core.runnables import RunnableLambda
    class DummyAIMessage(AIMessage):
        tool_calls = [] # Dummy for demonstration without actual tool calls

    def dummy_invoke(messages):
        # Simulate an LLM response based on user input
        last_user_message = next((m for m in reversed(messages) if isinstance(m, HumanMessage)), None)
        if last_user_message and "multiply" in last_user_message.content.lower():
            # Simulate a tool call if 'multiply' is mentioned
            return DummyAIMessage(
                content="Okay, let me calculate that for you.",
                tool_calls=[{"name": "multiply", "args": {"a": 5, "b": 3}}]
            )
        return DummyAIMessage(content=f"Hello, I received your message: '{last_user_message.content if last_user_message else 'None'}'. How can I help?")

    model = RunnableLambda(dummy_invoke).bind_tools(tools)


# 4. Define Graph Nodes

# Node for LLM interaction
def call_model(state: AgentState) -> AgentState:
    """Calls the LLM with the current messages and returns its response."""
    print(f"\n--- call_model node ---")
    print(f"Current messages before LLM call: {[m.content for m in state['messages']]}")
    print(f"Current context: {state.get('context', 'No context')}")

    # Add a system prompt for context
    system_prompt = SystemMessage(content="You are a helpful assistant. Use tools when appropriate.")
    messages_to_send = [system_prompt] + state['messages']

    response = model.invoke(messages_to_send)
    print(f"LLM response: {response.content}")
    return {"messages": [response]}

# Conditional edge logic
def should_continue(state: AgentState) -> str:
    """
    Determines whether to continue to tools or end the conversation.
    If the last message contains tool calls, continue; otherwise, end.
    """
    last_message = state['messages'][-1]
    if last_message.tool_calls:
        print(f"--- should_continue: detected tool calls, continuing to 'tools' ---")
        return "continue"
    else:
        print(f"--- should_continue: no tool calls, ending ---")
        return "end"

# 5. Build the LangGraph
graph = StateGraph(AgentState)

graph.add_node("llm_node", call_model)
tool_node = ToolNode(tools=tools)
graph.add_node("tool_executor", tool_node)

# Define the workflow
graph.add_edge(START, "llm_node")
graph.add_conditional_edges(
    "llm_node",
    should_continue,
    {"continue": "tool_executor", "end": END}
)
graph.add_edge("tool_executor", "llm_node") # After tool execution, go back to LLM for summary

app = graph.compile()

# Helper function to print the stream
def print_stream(stream):
    print("\n--- Stream Output ---")
    for s in stream:
        print(s)
        print("-" * 20)
    print("--- End Stream Output ---\n")

# --- Demonstrating injectState ---

print("--- Scenario 1: Starting fresh (no injectState) ---")
initial_input = {"messages": [HumanMessage(content="What is 7 + 8?")]}
stream1 = app.stream(initial_input, stream_mode="values")
print_stream(stream1)


print("\n--- Scenario 2: Starting with pre-defined messages using injectState ---")
# Here, we're providing an initial state with existing messages.
# The graph will start as if these messages were already exchanged.
pre_existing_messages = [
    HumanMessage(content="Hello there!"),
    AIMessage(content="Hi! How can I assist you today?"),
    HumanMessage(content="I need to know what 12 multiplied by 5 is.")
]
# injectState is typically used with `.invoke()` or `.stream()` as an argument
# or can be used with `.get_graph().invoke/stream` if you have a compiled graph
# to set the initial state.
# For app.stream, you can pass the initial state directly in the inputs dict
# This is functionally similar to injectState when starting from START.
stream2_input = {
    "messages": pre_existing_messages
}
stream2 = app.stream(stream2_input, stream_mode="values")
print_stream(stream2)


print("\n--- Scenario 3: Simulating resuming a conversation (injecting state mid-way) ---")
# Imagine we saved the state after the LLM made a tool call and before the tool executed.
# Now we want to resume and let the tool execute.
# For simplicity, we'll manually construct a state that looks like the graph
# just finished `llm_node` and is about to enter `tool_executor`.
# In a real scenario, you'd save the actual state from a previous run.
saved_partial_state = {
    "messages": [
        HumanMessage(content="Please multiply 10 by 20."),
        AIMessage(content="Okay, I can do that.", tool_calls=[{"name": "multiply", "args": {"a": 10, "b": 20}}])
    ],
    "context": "Resumed from a checkpoint after LLM decided to use a tool."
}

# When resuming from a specific node, you use the 'from' argument.
# When `from` is specified, `injectState` essentially acts as the starting point,
# overriding the default START node and starting execution from the specified node
# with the provided state.
# Note: For `app.stream`, you'd typically pass the state as the *initial* input.
# The concept of `injectState` is more explicit when building dynamic graphs
# or resuming from exact states in the state machine using `get_graph().invoke(input, config={'recursion_limit': N}, state=saved_state)`.
# For the `app.stream` method of a compiled graph, injecting a specific 'initial'
# state means simply providing it in the input dictionary. LangGraph will ensure
# the state is correctly processed from the START node.

print("This demonstration directly sets the input as a 'resumed' state for stream.")
print("In a real application, you'd load a complete state object and use it with app.invoke/stream directly.")

stream3_input = saved_partial_state # This acts as the injected state for a new stream
stream3 = app.stream(stream3_input, stream_mode="values")
print_stream(stream3)

print("\n--- Scenario 4: Injecting only context ---")
# Start a new conversation, but provide some background context directly via state.
stream4_input = {
    "messages": [HumanMessage(content="What's the capital of France?")],
    "context": "This user is interested in geography."
}
stream4 = app.stream(stream4_input, stream_mode="values")
print_stream(stream4)



In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage],add_messages]

@tool
def add(a: int, b: int) -> int:
    """This is an addition function that adds two integers."""
    return a + b
@tool
def sub(a,b):
    "this is a subtraction function"
    return a-b

@tool
def mult(a,b):
    'this is a multiplacation function'
    return a*b

@tool
def heater_energy(mass_flow:float,heat_cap:float,in_temp:float,out_temp:float):
    """heat energy if calculated in this function where  mass_flow is the mass of the material flow , heat_cap is the heat capacity of the materia
    ,in_temp is the inlet temperature , out_temp is the outlet temperature of the stream
    """
    return mass_flow*heat_cap*(out_temp-in_temp)

tools = [add,sub,mult,heater_energy]


llm=init_chat_model(model='llama3.2',model_provider="ollama",  # Explicitly state the provider
    temperature=0.7,).bind_tools(tools)



def model_call(state: AgentState) -> AgentState:
    """
    Invokes the language model with the current messages and returns its response.
    """
    system_prompt = SystemMessage(content='You are my AI assistance, please answer my query.')
    # Concatenate the system prompt with the current messages
    response = llm.invoke([system_prompt] + state['messages'])
    return {'messages': [response]}

def should_continue(state: AgentState) -> str:
    """
    Determines whether to continue to tools or end the conversation.
    If the last message contains tool calls, continue to tools; otherwise, end.
    """
    message = state['messages']
    last_message = message[-1]
    if not last_message.tool_calls:
        return 'end'
    else:
        return 'continue'




graph = StateGraph(AgentState)
graph.add_node('our_agent', model_call) # Node for LLM interaction
tool_node = ToolNode(tools=tools) # Node for tool execution
graph.add_node('tools', tool_node) # Node for tool execution
graph.add_edge(START, 'our_agent') # Start the graph by calling our agent
graph.add_conditional_edges(
    'our_agent',
    should_continue,
    {
        'continue': 'tools', # If tool calls are present, go to 'tools' node
        'end': END           # If no tool calls, end the graph
    },
)

graph.add_edge('tools', 'our_agent') # After tool execution, go back to 'our_agent'

# Compile the graph to create the runnable application
app = graph.compile()

# def print_stream(stream):
#     for s in stream :
#         message=s['messages'][-1]
#         if isinstance(message,tuple):
#             print(message)

#         else:
#             message.pretty_print()

def print_stream(stream):
    """
    Iterates through the stream of states and prints the last message in each state.
    """
    for s in stream:
        # Corrected typo: 'messsages' -> 'messages'
        message = s['messages'][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

# inputs={'messages':[{'user','Add 3 + 4.'}]}
# # inputs = {'messages': [HumanMessage(content="Add 3+4")]}
# # print_stream(app.stream(inputs,stream_mode='values'))
# print_stream(app.stream(inputs, stream_mode='values'))

# Example input for the graph
# Corrected message format: use HumanMessage object
inputs = {'messages': [HumanMessage(content="Add 3+4 , subtract 5 from 3 and multiply 15 times 10 , 10kg of water is entering a heater and the heat capacity of water is 4.18 it enters at 35 C and exit at 80C what is the energy produced")]}

# Run the application and print the stream
print_stream(app.stream(inputs, stream_mode='values'))


In [ ]:
def heater(inlet_stream:class,heater_type:str,condition:float):
    """Heater is a heating unit where the temperature of the inlet stream is increased 
    inlet_stream = stream class is sent as 
    heater_type=heater type based on data whether 'outlet_temperature' or 'temperature_difference',
    condition=value of outlet temperature or temperature difference 
    """
    heat_capacity=inlet_stream.cp()
    flow_rate=inlet_stream.flow()
    if heater_type =='outlet_temperature':
        temperature_diff=inlet_stream.T() - condition
    else:
        temperature_diff=condition
    energy =flow_rate*heat_capacity*temperature_diff

In [ ]:
class stream_spec():
    def __init__(self,name,Temperature:float,pressure:float,composition:list,flow_rate:list)
        self.name=name
        self.Temperature=Temperature
        self.pressure=pressure
        self.composition=composition
        self.flow_rate=flow_rate

def heater(name:str,inlet_stream:class,heater_type:str,condition:float):
    """Heater is a heating unit where the temperature of the inlet stream is increased 
    name = name of the heater unit 
    inlet_stream = stream class is sent as 
    heater_type=heater type based on data whether 'outlet_temperature' or 'temperature_difference',
    condition=value of outlet temperature or temperature difference 
    """
    flow_rate=inlet_stream.flow_rate()
    if heater_type =='outlet_temperature':
        temperature_out=inlet_stream.Temperature() - condition
    else:
        temperature_out=condition + condition
    heater=stream_spec(f'{name}_outlet',temperature_out,inlet_stream.pressure(),inlet_stream.composition(),inlet_stream.flow_rate())
     return heater

def pump(name:str,inlet_stream:class,heater_type:str,condition:float):
    """Heater is a heating unit where the temperature of the inlet stream is increased 
    name = name of the heater unit 
    inlet_stream = stream class is sent as 
    heater_type=heater type based on data whether 'outlet_temperature' or 'temperature_difference',
    condition=value of outlet temperature or temperature difference 
    """
    flow_rate=inlet_stream.flow_rate()
    if heater_type =='outlet_temperature':
        temperature_out=inlet_stream.Temperature() - condition
    else:
        temperature_out=condition + condition
    heater=stream_spec(f'{name}_outlet',temperature_out,inlet_stream.pressure(),inlet_stream.composition(),inlet_stream.flow_rate())
     return heater

In [ ]:
import csv
import io

class PropertyPackage:
    """
    A class to calculate ideal gas properties for a stream of chemicals.

    This class loads chemical data from a simulated database and provides
    methods to calculate properties for each component at a given
    temperature and pressure.
    """
    
    # R: Ideal Gas Constant in J/(mol*K)
    R = 8.314

    # Simulated chemical data as a CSV string
    # This data is a simplified example. In a real application, this would
    # be read from a file or a proper database.
    # The columns match the user's provided list.
    CHEMICAL_DATABASE_CSV = """
Name,Critical_Temperature,Critical_Temp_unit,Critical_Pressure,Critical_pressure_unit,Critical_Volume,Critical_vol_unit,Critical_Compressibility,Boiling_point,Boiling_point_unit,Mol_wt,Liquid_volume_at_BP,Liquid_vol_unit_at_BP,AcentricityFactor,Heat_of_Formation,Heat_of_Formation_unit,Gibbs_Energy_Of_Formation,Gibbs_Energy_Of_Formation_unit,Heat_of_combustion,Heat_of_combustion_unit,Lq_Den_A,Lq_Den_B,Lq_Den_C,Lq_Den_D,Lq_Den_Tmin,Lq_Den_Tmax,Lq_Den_unit,Vap_P_A,Vap_P_B,Vap_P_C,Vap_P_D,Vap_P_E,Vap_P_Tmin,Vap_P_Tmax,Vap_P_unit,Liq_Cp_A,Liq_Cp_B,Liq_Cp_C,Liq_Cp_D,Liq_Cp_E,Liq_Cp_Tmin,Liq_Cp_Tmax,Liq_Cp_unit,Gas_Cp_A,Gas_Cp_B,Gas_Cp_C,Gas_Cp_D,Gas_Cp_E,Gas_Cp_Tmin,Gas_Cp_Tmax,Gas_Cp_unit,Liq_vis_A,Liq_vis_B,Liq_vis_C,Liq_vis_D,Liq_vis_E,Liq_vis_Tmin,Liq_vis_Tmax,Liq_vis_unit,Liq_Therm_con_A,Liq_Therm_con_B,Liq_Therm_con_C,Liq_Therm_con_D,Liq_Therm_con_E,Liq_Therm_con_Tmin,Liq_Therm_con_Tmax,Liq_Therm_con_unit,Vap_Therm_con_A,Vap_Therm_con_B,Vap_Therm_con_C,Vap_Therm_con_D,Vap_Therm_con_Tmin,Vap_Therm_con_Tmax,Vap_Therm_con_unit
Methane,190.56,K,4.599,MPa,,,,,111.66,K,16.042,,,-0.011, -74.85,kJ/mol,,,,,145.42,1.385,258.96,,111.7,190.56,kg/m3,7.90422,10.66986,-0.4905,1.2584e-09,2,,,,,5.539,0.0305,0.0001,,298.15,1000,J/(mol K),34.331,-5.465e-02,3.159e-04,-1.491e-07,,298.15,1500,J/(mol K),,,,,,,,,,
Water,647.1,K,22.06,MPa,,,,,373.15,K,18.015,,,-0.344,-285.83,kJ/mol,,,,,999.0,0.000,273.15,373.15,kg/m3,8.07131,1730.63,233.426,0.0,,,,61.43,3.023E-02,0.0,0.0,0.0,273.15,373.15,J/(mol K),33.32,-0.2526e-02,-0.201e-05,0.002e-08,,298.15,1000,J/(mol K),,,,,,,,,,
"""

    def __init__(self, chemicals, temperature_K, pressure_Pa):
        """
        Initializes the property package with a list of chemicals, T, and P.

        Args:
            chemicals (list): A list of chemical names (e.g., ['Methane', 'Water']).
            temperature_K (float): The stream temperature in Kelvin.
            pressure_Pa (float): The stream pressure in Pascals.
        """
        self.chemicals = chemicals
        self.temperature_K = temperature_K
        self.pressure_Pa = pressure_Pa
        self.data = self._load_database()

    def _load_database(self):
        """
        Loads the chemical data from the internal CSV string into a dictionary.
        """
        data_dict = {}
        # Use io.StringIO to treat the string as a file
        reader = csv.DictReader(io.StringIO(self.CHEMICAL_DATABASE_CSV))
        for row in reader:
            data_dict[row['Name']] = row
        return data_dict

    def _get_data(self, chemical_name):
        """
        Helper method to retrieve data for a specific chemical.
        """
        if chemical_name in self.data:
            return self.data[chemical_name]
        else:
            raise ValueError(f"Chemical '{chemical_name}' not found in database.")

    def calculate_ideal_cp(self, chemical_name):
        """
        Calculates the ideal gas molar heat capacity (Cp) for a chemical.
        
        Formula: Cp = A + B*T + C*T^2 + D*T^3
        
        Returns:
            float: Molar heat capacity in J/(mol*K).
        """
        chem_data = self._get_data(chemical_name)
        T = self.temperature_K
        
        try:
            A = float(chem_data['Gas_Cp_A'])
            B = float(chem_data['Gas_Cp_B'])
            C = float(chem_data['Gas_Cp_C'])
            D = float(chem_data['Gas_Cp_D'])
            
            cp = A + B * T + C * (T**2) + D * (T**3)
            return cp
        except (ValueError, KeyError) as e:
            print(f"Error calculating Cp for {chemical_name}: {e}")
            return None

    def calculate_ideal_enthalpy(self, chemical_name, T_ref=298.15):
        """
        Calculates the ideal gas molar enthalpy (H) for a chemical relative to a
        reference temperature (T_ref) and the heat of formation.
        
        H = H_formation + integral(Cp dT) from T_ref to T_stream.
        
        Returns:
            float: Molar enthalpy in J/mol.
        """
        chem_data = self._get_data(chemical_name)
        T = self.temperature_K
        
        try:
            # Heat of formation in kJ/mol, converted to J/mol
            H_form = float(chem_data['Heat_of_Formation']) * 1000
            
            # Cp polynomial coefficients
            A = float(chem_data['Gas_Cp_A'])
            B = float(chem_data['Gas_Cp_B'])
            C = float(chem_data['Gas_Cp_C'])
            D = float(chem_data['Gas_Cp_D'])
            
            # Integral of Cp(T) from T_ref to T
            # integral(A + BT + CT^2 + DT^3) dT
            # = A*T + B*T^2/2 + C*T^3/3 + D*T^4/4
            sensible_heat = (
                A * (T - T_ref) +
                (B/2) * (T**2 - T_ref**2) +
                (C/3) * (T**3 - T_ref**3) +
                (D/4) * (T**4 - T_ref**4)
            )
            
            total_enthalpy = H_form + sensible_heat
            return total_enthalpy
            
        except (ValueError, KeyError) as e:
            print(f"Error calculating Enthalpy for {chemical_name}: {e}")
            return None

    def calculate_ideal_molar_volume(self):
        """
        Calculates the ideal gas molar volume using the ideal gas law (V = RT/P).
        
        Returns:
            float: Molar volume in m^3/mol.
        """
        return (self.R * self.temperature_K) / self.pressure_Pa

    def get_stream_properties(self):
        """
        Calculates and returns a dictionary of properties for each chemical
        in the stream.
        """
        stream_properties = {}
        for chem in self.chemicals:
            try:
                molar_volume = self.calculate_ideal_molar_volume()
                molar_cp = self.calculate_ideal_cp(chem)
                molar_enthalpy = self.calculate_ideal_enthalpy(chem)
                
                stream_properties[chem] = {
                    'Temperature (K)': self.temperature_K,
                    'Pressure (Pa)': self.pressure_Pa,
                    'Ideal Molar Volume (m^3/mol)': molar_volume,
                    'Ideal Molar Cp (J/mol.K)': molar_cp,
                    'Ideal Molar Enthalpy (J/mol)': molar_enthalpy
                }
            except ValueError as e:
                stream_properties[chem] = {'Error': str(e)}
        return stream_properties

# --- Example Usage ---
if __name__ == "__main__":
    # Define the stream conditions and components
    components = ['Methane', 'Water']
    temperature = 400.0  # K
    pressure = 101325.0  # Pa (1 atm)
    
    # Create an instance of the property package
    stream_package = PropertyPackage(components, temperature, pressure)
    
    # Get the calculated properties
    properties = stream_package.get_stream_properties()
    
    # Print the results
    print("--- Stream Properties Calculation ---")
    for chem_name, props in properties.items():
        print(f"\nProperties for {chem_name}:")
        for key, value in props.items():
            print(f"  {key}: {value:.4f}")
    
    print("\n-------------------------------------")



In [ ]:
import csv
import io
import math

class PropertyPackage:
    """
    A class to calculate ideal gas and VLE properties for a stream of chemicals.

    This class loads chemical data from a simulated database and provides
    methods to calculate properties like ideal gas heat capacity and enthalpy,
    as well as vapor-liquid equilibrium (VLE) properties using Raoult's Law.
    """
    
    # R: Ideal Gas Constant in J/(mol*K)
    R = 8.314

    # Simulated chemical data as a CSV string
    # This data is a simplified example. In a real application, this would
    # be read from a file or a proper database.
    # Vap_P_A, Vap_P_B, Vap_P_C are Antoine equation coefficients.
    CHEMICAL_DATABASE_CSV = """
Name,Critical_Temperature,Critical_Temp_unit,Critical_Pressure,Critical_pressure_unit,Critical_Volume,Critical_vol_unit,Critical_Compressibility,Boiling_point,Boiling_point_unit,Mol_wt,Liquid_volume_at_BP,Liquid_vol_unit_at_BP,AcentricityFactor,Heat_of_Formation,Heat_of_Formation_unit,Gibbs_Energy_Of_Formation,Gibbs_Energy_Of_Formation_unit,Heat_of_combustion,Heat_of_combustion_unit,Lq_Den_A,Lq_Den_B,Lq_Den_C,Lq_Den_D,Lq_Den_Tmin,Lq_Den_Tmax,Lq_Den_unit,Vap_P_A,Vap_P_B,Vap_P_C,Vap_P_D,Vap_P_E,Vap_P_Tmin,Vap_P_Tmax,Vap_P_unit,Liq_Cp_A,Liq_Cp_B,Liq_Cp_C,Liq_Cp_D,Liq_Cp_E,Liq_Cp_Tmin,Liq_Cp_Tmax,Liq_Cp_unit,Gas_Cp_A,Gas_Cp_B,Gas_Cp_C,Gas_Cp_D,Gas_Cp_E,Gas_Cp_Tmin,Gas_Cp_Tmax,Gas_Cp_unit,Liq_vis_A,Liq_vis_B,Liq_vis_C,Liq_vis_D,Liq_vis_E,Liq_vis_Tmin,Liq_vis_Tmax,Liq_vis_unit,Liq_Therm_con_A,Liq_Therm_con_B,Liq_Therm_con_C,Liq_Therm_con_D,Liq_Therm_con_E,Liq_Therm_con_Tmin,Liq_Therm_con_Tmax,Liq_Therm_con_unit,Vap_Therm_con_A,Vap_Therm_con_B,Vap_Therm_con_C,Vap_Therm_con_D,Vap_Therm_con_Tmin,Vap_Therm_con_Tmax,Vap_Therm_con_unit
Methane,190.56,K,4.599,MPa,,,,,111.66,K,16.042,,,-0.011,-74.85,kJ/mol,,,,,145.42,1.385,258.96,,111.7,190.56,kg/m3,6.61113,-641.527,1.2584e-09,2,0.0,,,,,5.539,0.0305,0.0001,,298.15,1000,J/(mol K),34.331,-5.465e-02,3.159e-04,-1.491e-07,,298.15,1500,J/(mol K),,,,,,,,,,
Water,647.1,K,22.06,MPa,,,,,373.15,K,18.015,,,-0.344,-285.83,kJ/mol,,,,,999.0,0.000,273.15,373.15,kg/m3,8.07131,1730.63,233.426,0.0,0.0,,,,,61.43,3.023E-02,0.0,0.0,0.0,273.15,373.15,J/(mol K),33.32,-0.2526e-02,-0.201e-05,0.002e-08,,298.15,1000,J/(mol K),,,,,,,,,,
"""

    def __init__(self, chemicals, temperature_K=None, pressure_Pa=None):
        """
        Initializes the property package with a list of chemicals, T, and P.

        Args:
            chemicals (list): A list of chemical names (e.g., ['Methane', 'Water']).
            temperature_K (float): The stream temperature in Kelvin.
            pressure_Pa (float): The stream pressure in Pascals.
        """
        self.chemicals = chemicals
        self.temperature_K = temperature_K
        self.pressure_Pa = pressure_Pa
        self.data = self._load_database()

    def _load_database(self):
        """
        Loads the chemical data from the internal CSV string into a dictionary.
        """
        data_dict = {}
        # Use io.StringIO to treat the string as a file
        reader = csv.DictReader(io.StringIO(self.CHEMICAL_DATABASE_CSV))
        for row in reader:
            data_dict[row['Name']] = row
        return data_dict

    def _get_data(self, chemical_name):
        """
        Helper method to retrieve data for a specific chemical.
        """
        if chemical_name in self.data:
            return self.data[chemical_name]
        else:
            raise ValueError(f"Chemical '{chemical_name}' not found in database.")

    def calculate_ideal_cp(self, chemical_name):
        """
        Calculates the ideal gas molar heat capacity (Cp) for a chemical.
        
        Formula: Cp = A + B*T + C*T^2 + D*T^3
        
        Returns:
            float: Molar heat capacity in J/(mol*K).
        """
        chem_data = self._get_data(chemical_name)
        T = self.temperature_K
        
        try:
            A = float(chem_data['Gas_Cp_A'])
            B = float(chem_data['Gas_Cp_B'])
            C = float(chem_data['Gas_Cp_C'])
            D = float(chem_data['Gas_Cp_D'])
            
            cp = A + B * T + C * (T**2) + D * (T**3)
            return cp
        except (ValueError, KeyError) as e:
            print(f"Error calculating Cp for {chemical_name}: {e}")
            return None

    def calculate_ideal_enthalpy(self, chemical_name, T_ref=298.15):
        """
        Calculates the ideal gas molar enthalpy (H) for a chemical relative to a
        reference temperature (T_ref) and the heat of formation.
        
        H = H_formation + integral(Cp dT) from T_ref to T_stream.
        
        Returns:
            float: Molar enthalpy in J/mol.
        """
        chem_data = self._get_data(chemical_name)
        T = self.temperature_K
        
        try:
            # Heat of formation in kJ/mol, converted to J/mol
            H_form = float(chem_data['Heat_of_Formation']) * 1000
            
            # Cp polynomial coefficients
            A = float(chem_data['Gas_Cp_A'])
            B = float(chem_data['Gas_Cp_B'])
            C = float(chem_data['Gas_Cp_C'])
            D = float(chem_data['Gas_Cp_D'])
            
            # Integral of Cp(T) from T_ref to T
            # integral(A + BT + CT^2 + DT^3) dT
            # = A*T + B*T^2/2 + C*T^3/3 + D*T^4/4
            sensible_heat = (
                A * (T - T_ref) +
                (B/2) * (T**2 - T_ref**2) +
                (C/3) * (T**3 - T_ref**3) +
                (D/4) * (T**4 - T_ref**4)
            )
            
            total_enthalpy = H_form + sensible_heat
            return total_enthalpy
            
        except (ValueError, KeyError) as e:
            print(f"Error calculating Enthalpy for {chemical_name}: {e}")
            return None

    def calculate_ideal_molar_volume(self):
        """
        Calculates the ideal gas molar volume using the ideal gas law (V = RT/P).
        
        Returns:
            float: Molar volume in m^3/mol.
        """
        if self.pressure_Pa == 0:
            raise ValueError("Pressure cannot be zero for ideal gas law calculation.")
        return (self.R * self.temperature_K) / self.pressure_Pa
        
    def calculate_vapor_pressure(self, chemical_name, temperature_K):
        """
        Calculates the vapor pressure (P_sat) for a chemical using the Antoine equation.
        
        Formula: log10(P_sat [bar]) = A - B / (T [C] + C)
        Database coefficients are modified for P in Pa and T in K.
        
        Returns:
            float: Vapor pressure in Pascals.
        """
        chem_data = self._get_data(chemical_name)
        
        try:
            # Coefficients are typically for log10(P) and T in Celsius, but we'll adapt.
            # Assuming the provided Vap_P coefficients are for a simple Antoine form
            # P_sat [Pa] = exp(A - B/T[K])
            A = float(chem_data['Vap_P_A'])
            B = float(chem_data['Vap_P_B'])
            
            # This is a simplified Antoine-like equation P_sat = exp(A - B/T)
            # A more accurate form would use the C and D coefficients.
            # The simplified form is sufficient for this example.
            
            P_sat_Pa = math.exp(A - B / temperature_K)
            return P_sat_Pa
            
        except (ValueError, KeyError) as e:
            raise ValueError(f"Error calculating vapor pressure for {chemical_name}: {e}")

    def calculate_bubble_point(self, mole_fractions, total_pressure_Pa):
        """
        Calculates the bubble point temperature using Raoult's Law and a bisection method.
        
        Raoult's Law: P = sum(x_i * P_sat_i(T))
        
        Args:
            mole_fractions (dict): A dictionary of liquid mole fractions {chem: x_i}.
            total_pressure_Pa (float): The total system pressure in Pascals.
            
        Returns:
            float: The bubble point temperature in Kelvin.
        """
        
        def bubble_point_function(T):
            """
            The function to find the root of: P - sum(x_i * P_sat_i(T)) = 0
            """
            sum_partial_pressures = 0
            for chem, x in mole_fractions.items():
                if x > 0:
                    P_sat = self.calculate_vapor_pressure(chem, T)
                    sum_partial_pressures += x * P_sat
            return total_pressure_Pa - sum_partial_pressures

        T_low = 100.0  # Initial low guess for temperature (K)
        T_high = 1000.0 # Initial high guess for temperature (K)
        tolerance = 1e-6
        max_iterations = 100

        for i in range(max_iterations):
            T_mid = (T_low + T_high) / 2
            f_mid = bubble_point_function(T_mid)

            if abs(f_mid) < tolerance:
                return T_mid
            
            f_low = bubble_point_function(T_low)
            if f_low * f_mid < 0:
                T_high = T_mid
            else:
                T_low = T_mid
                
        raise RuntimeError("Bisection method failed to converge for bubble point.")

    def calculate_dew_point(self, mole_fractions, total_pressure_Pa):
        """
        Calculates the dew point temperature using Raoult's Law and a bisection method.
        
        Raoult's Law: 1/P = sum(y_i / P_sat_i(T))
        
        Args:
            mole_fractions (dict): A dictionary of vapor mole fractions {chem: y_i}.
            total_pressure_Pa (float): The total system pressure in Pascals.
            
        Returns:
            float: The dew point temperature in Kelvin.
        """
        def dew_point_function(T):
            """
            The function to find the root of: 1/P - sum(y_i / P_sat_i(T)) = 0
            """
            sum_of_inverse_pressures = 0
            for chem, y in mole_fractions.items():
                if y > 0:
                    P_sat = self.calculate_vapor_pressure(chem, T)
                    sum_of_inverse_pressures += y / P_sat
            return (1 / total_pressure_Pa) - sum_of_inverse_pressures
            
        T_low = 100.0  # Initial low guess for temperature (K)
        T_high = 1000.0 # Initial high guess for temperature (K)
        tolerance = 1e-6
        max_iterations = 100

        for i in range(max_iterations):
            T_mid = (T_low + T_high) / 2
            f_mid = dew_point_function(T_mid)

            if abs(f_mid) < tolerance:
                return T_mid
            
            f_low = dew_point_function(T_low)
            if f_low * f_mid < 0:
                T_high = T_mid
            else:
                T_low = T_mid
                
        raise RuntimeError("Bisection method failed to converge for dew point.")
        
    def get_stream_properties(self):
        """
        Calculates and returns a dictionary of ideal gas properties for each chemical
        in the stream based on the provided T and P.
        """
        stream_properties = {}
        for chem in self.chemicals:
            try:
                molar_volume = self.calculate_ideal_molar_volume()
                molar_cp = self.calculate_ideal_cp(chem)
                molar_enthalpy = self.calculate_ideal_enthalpy(chem)
                
                stream_properties[chem] = {
                    'Temperature (K)': self.temperature_K,
                    'Pressure (Pa)': self.pressure_Pa,
                    'Ideal Molar Volume (m^3/mol)': molar_volume,
                    'Ideal Molar Cp (J/mol.K)': molar_cp,
                    'Ideal Molar Enthalpy (J/mol)': molar_enthalpy
                }
            except ValueError as e:
                stream_properties[chem] = {'Error': str(e)}
        return stream_properties

# --- Example Usage ---
if __name__ == "__main__":
    # --- Ideal Gas Properties Calculation ---
    print("--- Ideal Gas Properties Calculation ---")
    components = ['Methane', 'Water']
    temperature = 400.0  # K
    pressure = 101325.0  # Pa (1 atm)
    
    stream_package_ideal = PropertyPackage(components, temperature, pressure)
    properties = stream_package_ideal.get_stream_properties()
    
    for chem_name, props in properties.items():
        print(f"\nProperties for {chem_name}:")
        for key, value in props.items():
            print(f"  {key}: {value:.4f}")
    
    print("\n-------------------------------------")
    
    # --- VLE Properties Calculation ---
    print("--- VLE Properties Calculation (Raoult's Law) ---")
    components_vle = ['Water', 'Methane']
    liquid_mole_fractions = {'Water': 0.8, 'Methane': 0.2}
    vapor_mole_fractions = {'Water': 0.005, 'Methane': 0.995} # Note: This is just an example for demonstration.
    total_pressure = 101325.0 # Pa
    
    # Create an instance for VLE calculations
    stream_package_vle = PropertyPackage(components_vle)
    
    # Calculate bubble point temperature
    try:
        T_bubble = stream_package_vle.calculate_bubble_point(liquid_mole_fractions, total_pressure)
        print(f"\nBubble Point Temperature at {total_pressure:.0f} Pa:")
        print(f"  {T_bubble:.4f} K")
    except (ValueError, RuntimeError) as e:
        print(f"Error calculating bubble point: {e}")
        
    # Calculate dew point temperature
    try:
        T_dew = stream_package_vle.calculate_dew_point(vapor_mole_fractions, total_pressure)
        print(f"\nDew Point Temperature at {total_pressure:.0f} Pa:")
        print(f"  {T_dew:.4f} K")
    except (ValueError, RuntimeError) as e:
        print(f"Error calculating dew point: {e}")

    print("\n-------------------------------------")



In [ ]:
import csv
import io
import math
import pandas as pd

class PropertyPackage:
    """
    A class to calculate a wide range of properties for a stream of chemicals.

    This class loads chemical data from a simulated database and provides
    a suite of methods to calculate ideal gas, liquid, and VLE properties
    for each component at a given temperature and pressure.
    """
    
    # R: Ideal Gas Constant in J/(mol*K)
    R = 8.314

    # Simulated chemical data as a CSV string
    # This data is a simplified example. In a real application, this would
    # be read from a file or a proper database.
    # A more complete database would have all coefficients for all chemicals.
    self.CHEMICAL_DATABASE_CSV = 'Open_soruce_db.csv'

    def __init__(self, chemicals, temperature_K=None, pressure_Pa=None):
        """
        Initializes the property package with a list of chemicals, T, and P.

        Args:
            chemicals (list): A list of chemical names (e.g., ['Methane', 'Water']).
            temperature_K (float): The stream temperature in Kelvin.
            pressure_Pa (float): The stream pressure in Pascals.
        """
        self.chemicals = chemicals
        self.temperature_K = temperature_K
        self.pressure_Pa = pressure_Pa
        self.data = pd.read_csv('Open_soruce_db.csv')

    # def _load_database(self):
    #     """
    #     Loads the chemical data from the internal CSV string into a dictionary.
    #     """
    #     # data_dict = {}
    #     # # Use io.StringIO to treat the string as a file
    #     # reader = csv.DictReader(io.StringIO(self.CHEMICAL_DATABASE_CSV))
    #     # for row in reader:
    #     #     data_dict[row['Name']] = row
    #     data=pd.read_csv('Open_soruce_db.csv')
    #     return data

    def _get_data(self, chemical_name):
        """
        Helper method to retrieve data for a specific chemical.
        """
        if chemical_name in self.data:
            return self.data[chemical_name]
        else:
            raise ValueError(f"Chemical '{chemical_name}' not found in database.")

    def calculate_ideal_gas_cp(self, chemical_name):
        """
        Calculates the ideal gas molar heat capacity (Cp) for a chemical using a polynomial.
        Formula: Cp = A + B*T + C*T^2 + D*T^3 + E*T^4
        
        Returns:
            float: Molar heat capacity in J/(mol*K).
        """
        chem_data = self._get_data(chemical_name)
        T = self.temperature_K
        
        try:
            A = float(chem_data['Gas_Cp_A'])
            B = float(chem_data['Gas_Cp_B'])
            C = float(chem_data['Gas_Cp_C'])
            D = float(chem_data['Gas_Cp_D'])
            # The 'E' coefficient may be missing, so we handle it gracefully.
            E = float(chem_data.get('Gas_Cp_E', 0.0))
            
            cp = A + B * T + C * (T**2) + D * (T**3) + E * (T**4)
            return cp
        except (ValueError, KeyError) as e:
            print(f"Error calculating Gas Cp for {chemical_name}: {e}. Check if coefficients are present and valid.")
            return None

    def calculate_ideal_enthalpy(self, chemical_name, T_ref=298.15):
        """
        Calculates the ideal gas molar enthalpy (H) for a chemical relative to a
        reference temperature (T_ref) and the heat of formation.
        
        H = H_formation + integral(Cp dT) from T_ref to T_stream.
        
        Returns:
            float: Molar enthalpy in J/mol.
        """
        chem_data = self._get_data(chemical_name)
        T = self.temperature_K
        
        try:
            # Heat of formation in kJ/mol, converted to J/mol
            H_form = float(chem_data['Heat_of_Formation']) * 1000
            
            # Cp polynomial coefficients
            A = float(chem_data['Gas_Cp_A'])
            B = float(chem_data['Gas_Cp_B'])
            C = float(chem_data['Gas_Cp_C'])
            D = float(chem_data['Gas_Cp_D'])
            E = float(chem_data.get('Gas_Cp_E', 0.0))
            
            # Integral of Cp(T) from T_ref to T
            sensible_heat = (
                A * (T - T_ref) +
                (B/2) * (T**2 - T_ref**2) +
                (C/3) * (T**3 - T_ref**3) +
                (D/4) * (T**4 - T_ref**4) +
                (E/5) * (T**5 - T_ref**5)
            )
            
            total_enthalpy = H_form + sensible_heat
            return total_enthalpy
            
        except (ValueError, KeyError) as e:
            print(f"Error calculating Enthalpy for {chemical_name}: {e}. Check if coefficients are present and valid.")
            return None

    def calculate_ideal_molar_volume(self):
        """
        Calculates the ideal gas molar volume using the ideal gas law (V = RT/P).
        
        Returns:
            float: Molar volume in m^3/mol.
        """
        if self.pressure_Pa == 0:
            raise ValueError("Pressure cannot be zero for ideal gas law calculation.")
        return (self.R * self.temperature_K) / self.pressure_Pa
        
    def calculate_vapor_pressure(self, chemical_name, temperature_K):
        """
        Calculates the vapor pressure (P_sat) for a chemical using an Antoine-like equation.
        Formula: ln(P_sat [Pa]) = A + B/T[K] + C*ln(T[K]) + D*T[K]**E
        
        Returns:
            float: Vapor pressure in Pascals.
        """
        chem_data = self._get_data(chemical_name)
        
        try:
            A = float(chem_data['Vap_P_A'])
            B = float(chem_data['Vap_P_B'])
            C = float(chem_data.get('Vap_P_C', 0.0))
            D = float(chem_data.get('Vap_P_D', 0.0))
            E = float(chem_data.get('Vap_P_E', 1.0)) # E is often 1 or 2 for common equations
            
            # A simplified Antoine-like equation
            log_P_sat = A + B/temperature_K + C * math.log(temperature_K) + D * (temperature_K**E)
            P_sat_Pa = math.exp(log_P_sat)
            return P_sat_Pa
            
        except (ValueError, KeyError) as e:
            raise ValueError(f"Error calculating vapor pressure for {chemical_name}: {e}")

    def calculate_liquid_density(self, chemical_name):
        """
        Calculates the liquid density (rho) for a chemical using a polynomial.
        Formula: rho = A + B*T + C*T^2 + D*T^3
        
        Returns:
            float: Liquid density in kg/m^3.
        """
        chem_data = self._get_data(chemical_name)
        T = self.temperature_K
        
        try:
            A = float(chem_data['Lq_Den_A'])
            B = float(chem_data['Lq_Den_B'])
            C = float(chem_data.get('Lq_Den_C', 0.0))
            D = float(chem_data.get('Lq_Den_D', 0.0))
            
            density = A + B * T + C * (T**2) + D * (T**3)
            return density
        except (ValueError, KeyError) as e:
            print(f"Error calculating Liquid Density for {chemical_name}: {e}. Check if coefficients are present and valid.")
            return None
            
    def calculate_liquid_cp(self, chemical_name):
        """
        Calculates the liquid molar heat capacity (Cp) for a chemical using a polynomial.
        Formula: Cp = A + B*T + C*T^2 + D*T^3 + E*T^4
        
        Returns:
            float: Liquid molar heat capacity in J/(mol*K).
        """
        chem_data = self._get_data(chemical_name)
        T = self.temperature_K
        
        try:
            A = float(chem_data['Liq_Cp_A'])
            B = float(chem_data['Liq_Cp_B'])
            C = float(chem_data['Liq_Cp_C'])
            D = float(chem_data.get('Liq_Cp_D', 0.0))
            E = float(chem_data.get('Liq_Cp_E', 0.0))
            
            cp = A + B * T + C * (T**2) + D * (T**3) + E * (T**4)
            return cp
        except (ValueError, KeyError) as e:
            print(f"Error calculating Liquid Cp for {chemical_name}: {e}. Check if coefficients are present and valid.")
            return None
    
    def calculate_bubble_point(self, mole_fractions, total_pressure_Pa):
        """
        Calculates the bubble point temperature using Raoult's Law and a bisection method.
        
        Raoult's Law: P = sum(x_i * P_sat_i(T))
        
        Args:
            mole_fractions (dict): A dictionary of liquid mole fractions {chem: x_i}.
            total_pressure_Pa (float): The total system pressure in Pascals.
            
        Returns:
            float: The bubble point temperature in Kelvin.
        """
        def bubble_point_function(T):
            sum_partial_pressures = 0
            for chem, x in mole_fractions.items():
                if x > 0:
                    P_sat = self.calculate_vapor_pressure(chem, T)
                    sum_partial_pressures += x * P_sat
            return total_pressure_Pa - sum_partial_pressures

        T_low = 100.0
        T_high = 1000.0
        tolerance = 1e-6
        max_iterations = 100

        for i in range(max_iterations):
            T_mid = (T_low + T_high) / 2
            f_mid = bubble_point_function(T_mid)

            if abs(f_mid) < tolerance:
                return T_mid
            
            f_low = bubble_point_function(T_low)
            if f_low * f_mid < 0:
                T_high = T_mid
            else:
                T_low = T_mid
                
        raise RuntimeError("Bisection method failed to converge for bubble point.")

    def calculate_dew_point(self, mole_fractions, total_pressure_Pa):
        """
        Calculates the dew point temperature using Raoult's Law and a bisection method.
        
        Raoult's Law: 1/P = sum(y_i / P_sat_i(T))
        
        Args:
            mole_fractions (dict): A dictionary of vapor mole fractions {chem: y_i}.
            total_pressure_Pa (float): The total system pressure in Pascals.
            
        Returns:
            float: The dew point temperature in Kelvin.
        """
        def dew_point_function(T):
            sum_of_inverse_pressures = 0
            for chem, y in mole_fractions.items():
                if y > 0:
                    P_sat = self.calculate_vapor_pressure(chem, T)
                    sum_of_inverse_pressures += y / P_sat
            return (1 / total_pressure_Pa) - sum_of_inverse_pressures
            
        T_low = 100.0
        T_high = 1000.0
        tolerance = 1e-6
        max_iterations = 100

        for i in range(max_iterations):
            T_mid = (T_low + T_high) / 2
            f_mid = dew_point_function(T_mid)

            if abs(f_mid) < tolerance:
                return T_mid
            
            f_low = dew_point_function(T_low)
            if f_low * f_mid < 0:
                T_high = T_mid
            else:
                T_low = T_mid
                
        raise RuntimeError("Bisection method failed to converge for dew point.")
        
    def get_stream_properties(self):
        """
        Calculates and returns a dictionary of all possible properties for each chemical
        in the stream based on the provided T and P.
        """
        stream_properties = {}
        for chem in self.chemicals:
            chem_props = {}
            chem_props['Temperature (K)'] = self.temperature_K
            chem_props['Pressure (Pa)'] = self.pressure_Pa
            
            try:
                chem_props['Mol_wt'] = float(self._get_data(chem).get('Mol_wt', 'N/A'))
            except (ValueError, TypeError):
                chem_props['Mol_wt'] = 'N/A'
            
            # Dynamically calculate properties if the methods and coefficients exist
            try:
                chem_props['Ideal Molar Volume (m^3/mol)'] = self.calculate_ideal_molar_volume()
            except ValueError as e:
                chem_props['Ideal Molar Volume (m^3/mol)'] = f'Error: {e}'

            try:
                chem_props['Ideal Gas Molar Cp (J/mol.K)'] = self.calculate_ideal_gas_cp(chem)
            except (ValueError, KeyError, TypeError):
                chem_props['Ideal Gas Molar Cp (J/mol.K)'] = 'N/A'

            try:
                chem_props['Ideal Molar Enthalpy (J/mol)'] = self.calculate_ideal_enthalpy(chem)
            except (ValueError, KeyError, TypeError):
                chem_props['Ideal Molar Enthalpy (J/mol)'] = 'N/A'

            try:
                chem_props['Liquid Density (kg/m^3)'] = self.calculate_liquid_density(chem)
            except (ValueError, KeyError, TypeError):
                chem_props['Liquid Density (kg/m^3)'] = 'N/A'

            try:
                chem_props['Liquid Molar Cp (J/mol.K)'] = self.calculate_liquid_cp(chem)
            except (ValueError, KeyError, TypeError):
                chem_props['Liquid Molar Cp (J/mol.K)'] = 'N/A'
            
            stream_properties[chem] = chem_props

        return stream_properties

# --- Example Usage ---
if __name__ == "__main__":
    # Define the stream conditions and components
    components = ['Methane', 'Water']
    temperature = 350.0  # K
    pressure = 101325.0  # Pa (1 atm)
    
    # Create an instance of the property package
    stream_package = PropertyPackage(components, temperature, pressure)
    
    # Get all calculated properties
    properties = stream_package.get_stream_properties()
    
    # Print the results
    print("--- Comprehensive Stream Properties Calculation ---")
    for chem_name, props in properties.items():
        print(f"\nProperties for {chem_name}:")
        for key, value in props.items():
            if isinstance(value, float):
                print(f"  {key}: {value:.4f}")
            else:
                print(f"  {key}: {value}")
    
    print("\n-------------------------------------")
    
    # --- VLE Properties Calculation ---
    print("--- VLE Properties Calculation (Raoult's Law) ---")
    components_vle = ['Water', 'Methane']
    liquid_mole_fractions = {'Water': 0.8, 'Methane': 0.2}
    vapor_mole_fractions = {'Water': 0.005, 'Methane': 0.995}
    total_pressure = 101325.0 # Pa
    
    # Create an instance for VLE calculations
    stream_package_vle = PropertyPackage(components_vle)
    
    try:
        T_bubble = stream_package_vle.calculate_bubble_point(liquid_mole_fractions, total_pressure)
        print(f"\nBubble Point Temperature at {total_pressure:.0f} Pa:")
        print(f"  {T_bubble:.4f} K")
    except (ValueError, RuntimeError) as e:
        print(f"Error calculating bubble point: {e}")
        
    try:
        T_dew = stream_package_vle.calculate_dew_point(vapor_mole_fractions, total_pressure)
        print(f"\nDew Point Temperature at {total_pressure:.0f} Pa:")
        print(f"  {T_dew:.4f} K")
    except (ValueError, RuntimeError) as e:
        print(f"Error calculating dew point: {e}")

    print("\n-------------------------------------")


In [4]:
dd=pd.read_csv('Open_source_db2.csv').columns
for i in dd:
    print(i)

Name
Critical_Temperature
Critical_Temp_unit
Critical_Pressure
Critical_pressure_unit
Critical_Volume
Critical_vol_unit
Critical_Compressibility
Boiling_point
Boiling_point_unit
Mol_wt
Liquid_volume_at_BP
Liquid_vol_unit_at_BP
AcentricityFactor
Heat_of_Formation
Heat_of_Formation_unit
Gibbs_Energy_Of_Formation
Gibbs_Energy_Of_Formation_unit
Heat_of_combustion
Heat_of_combustion_unit
Lq_Den_A
Lq_Den_B
Lq_Den_C
Lq_Den_D
Lq_Den_Tmin
Lq_Den_Tmax
Lq_Den_unit
Vap_P_A
Vap_P_B
Vap_P_C
Vap_P_D
Vap_P_E
Vap_P_Tmin
Vap_P_Tmax
Vap_P_unit
Liq_Cp_A
Liq_Cp_B
Liq_Cp_C
Liq_Cp_D
Liq_Cp_E
Liq_Cp_Tmin
Liq_Cp_Tmax
Liq_Cp_unit
Gas_Cp_A
Gas_Cp_B
Gas_Cp_C
Gas_Cp_D
Gas_Cp_E
Gas_Cp_Tmin
Gas_Cp_Tmax
Gas_Cp_unit
Liq_vis_A
Liq_vis_B
Liq_vis_C
Liq_vis_D
Liq_vis_E
Liq_vis_Tmin
Liq_vis_Tmax
Liq_vis_unit
Liq_Therm_con_A
Liq_Therm_con_B
Liq_Therm_con_C
Liq_Therm_con_D
Liq_Therm_con_E
Liq_Therm_con_Tmin
Liq_Therm_con_Tmax
Liq_Therm_con_unit
Vap_Therm_con_A
Vap_Therm_con_B
Vap_Therm_con_C
Vap_Therm_con_D
Vap_Therm_con_Tmi

In [ ]:
import csv
import io
import math
import pandas as pd

class PropertyPackage:
    """
    A class to calculate a wide range of properties for a stream of chemicals.

    This class loads chemical data from a database and provides methods to
    calculate individual and mixture properties for a given stream at a
    specific temperature and pressure, including phase determination.
    """
    
    # R: Ideal Gas Constant in J/(mol*K)
    R = 8.314
    
    def __init__(self,stream_name, chemicals_with_fractions, temperature_K, pressure_Pa, molar_flowrate_mol_s):
        """
        Initializes the property package with a dictionary of chemicals, T, P, and flow rate.

        Args:
            chemicals_with_fractions (dict): A dictionary of chemical names and their
                                             mole fractions (e.g., {'Methane': 0.8, 'Water': 0.2}).
            temperature_K (float): The stream temperature in Kelvin.
            pressure_Pa (float): The stream pressure in Pascals.
            molar_flowrate_mol_s (float): The total molar flow rate of the stream in mol/s.
        """
        # Store the chemical names and mole fractions
        self.stream_name = stream_name
        self.chemicals = list(chemicals_with_fractions.keys())
        self.mole_fractions = chemicals_with_fractions
        self.temperature_K = temperature_K
        self.pressure_Pa = pressure_Pa
        self.molar_flowrate_mol_s = molar_flowrate_mol_s
        
        # Load data from the database using pandas as requested
        # Note: You'll need an 'Open_soruce_db.csv' file in the same directory
        # In a real application, you might use a more robust database connection
        try:
            self.data = pd.read_csv('Open_soruce_db.csv')
        except FileNotFoundError:
            print("Warning: 'Open_soruce_db.csv' not found. Using a simulated database.")
            # Use the in-memory data if the file is not found
            self.data = self._load_simulated_database()

    def _load_simulated_database(self):
        """
        Loads the simulated chemical data from the internal CSV string into a pandas DataFrame.
        This is a fallback if the external CSV is not present.
        """
        CHEMICAL_DATABASE_CSV = """
Name,Critical_Temperature,Critical_Temp_unit,Critical_Pressure,Critical_pressure_unit,Critical_Volume,Critical_vol_unit,Critical_Compressibility,Boiling_point,Boiling_point_unit,Mol_wt,Liquid_volume_at_BP,Liquid_vol_unit_at_BP,AcentricityFactor,Heat_of_Formation,Heat_of_Formation_unit,Gibbs_Energy_Of_Formation,Gibbs_Energy_Of_Formation_unit,Heat_of_combustion,Heat_of_combustion_unit,Lq_Den_A,Lq_Den_B,Lq_Den_C,Lq_Den_D,Lq_Den_Tmin,Lq_Den_Tmax,Lq_Den_unit,Vap_P_A,Vap_P_B,Vap_P_C,Vap_P_D,Vap_P_E,Vap_P_Tmin,Vap_P_Tmax,Vap_P_unit,Liq_Cp_A,Liq_Cp_B,Liq_Cp_C,Liq_Cp_D,Liq_Cp_E,Liq_Cp_Tmin,Liq_Cp_Tmax,Liq_Cp_unit,Gas_Cp_A,Gas_Cp_B,Gas_Cp_C,Gas_Cp_D,Gas_Cp_E,Gas_Cp_Tmin,Gas_Cp_Tmax,Gas_Cp_unit,Liq_vis_A,Liq_vis_B,Liq_vis_C,Liq_vis_D,Liq_vis_E,Liq_vis_Tmin,Liq_vis_Tmax,Liq_vis_unit,Liq_Therm_con_A,Liq_Therm_con_B,Liq_Therm_con_C,Liq_Therm_con_D,Liq_Therm_con_E,Liq_Therm_con_Tmin,Liq_Therm_con_Tmax,Liq_Therm_con_unit,Vap_Therm_con_A,Vap_Therm_con_B,Vap_Therm_con_C,Vap_Therm_con_D,Vap_Therm_con_Tmin,Vap_Therm_con_Tmax,Vap_Therm_con_unit,Heat_of_vap_A,Heat_of_vap_B,Heat_of_vap_C,Heat_of_vap_D,Heat_of_vap_E,Heat_of_vap_Tmin,Heat_of_vap_Tmax,Heat_of_vap_unit
Methane,190.56,K,4.599,MPa,,,,,111.66,K,16.042,,,-0.011,-74.85,kJ/mol,,,,,145.42,1.385,258.96,,111.7,190.56,kg/m3,6.61113,-641.527,1.2584e-09,2,0.0,,,,,5.539,0.0305,0.0001,,298.15,1000,J/(mol K),34.331,-5.465e-02,3.159e-04,-1.491e-07,,298.15,1500,J/(mol K),,,,,,,,,,,0.005574,0.01633,0.000032,0.0,,,,0.00396,0.000045,0.0000000003,,200,600,W/m.K,242.0,-2.36,0.0,0.0,0.0,111.66,190.56,J/mol
Water,647.1,K,22.06,MPa,,,,,373.15,K,18.015,,,-0.344,-285.83,kJ/mol,,,,,999.0,0.000,273.15,373.15,kg/m3,8.07131,1730.63,233.426,0.0,0.0,,,,,61.43,3.023E-02,0.0,0.0,0.0,273.15,373.15,J/(mol K),33.32,-0.2526e-02,-0.201e-05,0.002e-08,,298.15,1000,J/(mol K),,,,,,,,,,,0.5714,0.001764,0.0,-0.00000185,0.0,273.15,373.15,W/m.K,0.0005,0.000033,-0.0000000003,,273.15,647.1,W/m.K,45000.0,-50.0,0.0,0.0,0.0,273.15,647.1,J/mol
"""
        return pd.read_csv(io.StringIO(CHEMICAL_DATABASE_CSV))

    def _get_data(self):
        """
        Helper method to retrieve data for a specific chemical from the DataFrame.
        Returns a pandas Series for the specified chemical's row.
        """
        try:
            return self.data.loc[self.data['Name'] == self.chemical_name].iloc[0]
        except IndexError:
            raise ValueError(f"Chemical '{self.chemical_name}' not found in database.")

    # --- Individual Property Calculation Methods ---

    def calculate_ideal_gas_cp(self):
        """
        Calculates the ideal gas molar heat capacity (Cp) for a chemical using a polynomial.
        Formula: Cp = A + B*T + C*T^2 + D*T^3 + E*T^4
        
        Returns:
            float: Molar heat capacity in J/(mol*K).
        """
        chem_data = self._get_data(self.chemical_name)
        T = self.temperature_K
        
        try:
            A = float(chem_data['Gas_Cp_A'])
            B = float(chem_data['Gas_Cp_B'])
            C = float(chem_data['Gas_Cp_C'])
            D = float(chem_data['Gas_Cp_D'])
            E = float(chem_data.get('Gas_Cp_E', 0.0))
            
            cp = A + B * T + C * (T**2) + D * (T**3) + E * (T**4)
            return cp
        except (ValueError, KeyError) as e:
            return None

    def calculate_ideal_enthalpy(self, T_ref=298.15):
        """
        Calculates the ideal gas molar enthalpy (H) for a chemical relative to a
        reference temperature (T_ref) and the heat of formation.
        
        Returns:
            float: Molar enthalpy in J/mol.
        """
        chem_data = self._get_data(self.chemical_name)
        T = self.temperature_K
        
        try:
            H_form = float(chem_data['Heat_of_Formation']) * 1000
            A = float(chem_data['Gas_Cp_A'])
            B = float(chem_data['Gas_Cp_B'])
            C = float(chem_data['Gas_Cp_C'])
            D = float(chem_data['Gas_Cp_D'])
            E = float(chem_data.get('Gas_Cp_E', 0.0))
            
            sensible_heat = (
                A * (T - T_ref) +
                (B/2) * (T**2 - T_ref**2) +
                (C/3) * (T**3 - T_ref**3) +
                (D/4) * (T**4 - T_ref**4) +
                (E/5) * (T**5 - T_ref**5)
            )
            
            total_enthalpy = H_form + sensible_heat
            return total_enthalpy
            
        except (ValueError, KeyError) as e:
            return None

    def calculate_liquid_enthalpy(self, T_ref=298.15):
        """
        Calculates the liquid molar enthalpy (H) for a chemical relative to a
        reference temperature (T_ref) and the heat of formation.
        
        Returns:
            float: Molar enthalpy in J/mol.
        """
        chem_data = self._get_data(self.chemical_name)
        T = self.temperature_K
        
        try:
            H_form = float(chem_data['Heat_of_Formation']) * 1000
            A = float(chem_data['Liq_Cp_A'])
            B = float(chem_data['Liq_Cp_B'])
            C = float(chem_data['Liq_Cp_C'])
            D = float(chem_data.get('Liq_Cp_D', 0.0))
            E = float(chem_data.get('Liq_Cp_E', 0.0))
            
            sensible_heat = (
                A * (T - T_ref) +
                (B/2) * (T**2 - T_ref**2) +
                (C/3) * (T**3 - T_ref**3) +
                (D/4) * (T**4 - T_ref**4) +
                (E/5) * (T**5 - T_ref**5)
            )
            
            total_enthalpy = H_form + sensible_heat
            return total_enthalpy
            
        except (ValueError, KeyError) as e:
            return None

    def calculate_vapor_pressure(self):
        """
        Calculates the vapor pressure (P_sat) for a chemical using an Antoine-like equation.
        
        Returns:
            float: Vapor pressure in Pascals.
        """
        chem_data = self._get_data(self.chemical_name)
        
        try:
            A = float(chem_data['Vap_P_A'])
            B = float(chem_data['Vap_P_B'])
            C = float(chem_data.get('Vap_P_C', 0.0))
            D = float(chem_data.get('Vap_P_D', 0.0))
            E = float(chem_data.get('Vap_P_E', 1.0))
            
            log_P_sat = A + B/self.temperature_K + C * math.log(self.temperature_K) + D * (self.temperature_K**E)
            P_sat_Pa = math.exp(log_P_sat)
            return P_sat_Pa
            
        except (ValueError, KeyError) as e:
            raise ValueError(f"Error calculating vapor pressure for {self.chemical_name}: {e}")

    def calculate_liquid_density(self):

        """
        Calculates the liquid density (rho) for a chemical using a polynomial.
        Formula: rho = A + B*T + C*T^2 + D*T^3
        
        Returns:
            float: Liquid density in kg/m^3.
        """
        chem_data = self._get_data(self.chemical_name)
        T = self.temperature_K
        
        try:
            A = float(chem_data['Lq_Den_A'])
            B = float(chem_data['Lq_Den_B'])
            C = float(chem_data.get('Lq_Den_C', 0.0))
            D = float(chem_data.get('Lq_Den_D', 0.0))
            
            density = A + B * T + C * (T**2) + D * (T**3)
            return density
        except (ValueError, KeyError) as e:
            return None
        
    def calculate_vapor_density(self):
        """
        Calculates the vapor density (rho) for a chemical using ideal gas law.
        Formula: pv=nRT => rho = P/(R*T)
        
        Returns:
            float: Liquid density in kg/m^3.
        """
        
        T = self.temperature_K
        
        try:
            density = self.pressure_Pa / (self.R * T)  # Ideal gas law rearranged for density
            # density *= 1000  # Convert from mol/m^3 to kg/m^3
            return density
        except (ValueError, KeyError) as e:
            return None
            
    def calculate_liquid_cp(self):
        """
        Calculates the liquid molar heat capacity (Cp) for a chemical using a polynomial.
        
        Returns:
            float: Liquid molar heat capacity in J/(mol*K).
        """
        chem_data = self._get_data(self.chemical_name)
        T = self.temperature_K
        
        try:
            A = float(chem_data['Liq_Cp_A'])
            B = float(chem_data['Liq_Cp_B'])
            C = float(chem_data['Liq_Cp_C'])
            D = float(chem_data.get('Liq_Cp_D', 0.0))
            E = float(chem_data.get('Liq_Cp_E', 0.0))
            
            cp = A + B * T + C * (T**2) + D * (T**3) + E * (T**4)
            return cp
        except (ValueError, KeyError) as e:
            return None

    def _calculate_molar_heat_of_vaporization(self):
        """
        Calculates the molar heat of vaporization (H_vap) for a chemical using a polynomial.
        
        Returns:
            float: Molar heat of vaporization in J/mol.
        """
        chem_data = self._get_data(self.chemical_name)
        T = self.temperature_K
        
        try:
            A = float(chem_data['Heat_of_vap_A'])
            B = float(chem_data['Heat_of_vap_B'])
            C = float(chem_data.get('Heat_of_vap_C', 0.0))
            D = float(chem_data.get('Heat_of_vap_D', 0.0))
            E = float(chem_data.get('Heat_of_vap_E', 0.0))
            
            h_vap = A + B * T + C * (T**2) + D * (T**3) + E * (T**4)
            return h_vap
        except (ValueError, KeyError) as e:
            return None

    def _calculate_liquid_thermal_conductivity(self):
        """
        Calculates the liquid thermal conductivity (k) for a chemical using a polynomial.
        Formula: k = A + B*T + C*T^2 + D*T^3 + E*T^4
        
        Returns:
            float: Thermal conductivity in W/(m*K).
        """
        chem_data = self._get_data(self.chemical_name)
        T = self.temperature_K
        
        try:
            A = float(chem_data['Liq_Therm_con_A'])
            B = float(chem_data['Liq_Therm_con_B'])
            C = float(chem_data['Liq_Therm_con_C'])
            D = float(chem_data.get('Liq_Therm_con_D', 0.0))
            E = float(chem_data.get('Liq_Therm_con_E', 0.0))
            
            k_liq = A + B * T + C * (T**2) + D * (T**3) + E * (T**4)
            return k_liq
        except (ValueError, KeyError) as e:
            return None

    def _calculate_vapor_thermal_conductivity(self):
        """
        Calculates the vapor thermal conductivity (k) for a chemical using a polynomial.
        Formula: k = A + B*T + C*T^2 + D*T^3
        
        Returns:
            float: Thermal conductivity in W/(m*K).
        """
        chem_data = self._get_data(self.chemical_name)
        T = self.temperature_K
        
        try:
            A = float(chem_data['Vap_Therm_con_A'])
            B = float(chem_data['Vap_Therm_con_B'])
            C = float(chem_data['Vap_Therm_con_C'])
            D = float(chem_data.get('Vap_Therm_con_D', 0.0))
            
            k_vap = A + B * T + C * (T**2) + D * (T**3)
            return k_vap
        except (ValueError, KeyError) as e:
            return None
    
    # --- Mixture Property Calculation Methods ---

    def calculate_mixture_molar_weight(self):
        """
        Calculates the mixture's molar weight based on a weighted average of components.
        
        Returns:
            float: The molar weight of the mixture in g/mol.
        """
        mixture_mw = 0.0
        for chem, x in self.mole_fractions.items():
            chem_data = self._get_data(chem)
            mw = float(chem_data['Mol_wt'])
            mixture_mw += x * mw
        return mixture_mw

    def _calculate_mixture_liquid_cp(self):
        """
        Calculates the mixture's liquid heat capacity based on a weighted average.
        
        Returns:
            float: The mixture's liquid heat capacity in J/(mol*K).
        """
        mixture_cp = 0.0
        for chem, x in self.mole_fractions.items():
            individual_cp = self.calculate_liquid_cp(chem, self.temperature_K)
            if individual_cp is not None:
                mixture_cp += x * individual_cp
        return mixture_cp

    def _calculate_mixture_gas_cp(self):
        """
        Calculates the mixture's ideal gas heat capacity based on a weighted average.
        
        Returns:
            float: The mixture's ideal gas heat capacity in J/(mol*K).
        """
        mixture_cp = 0.0
        for chem, x in self.mole_fractions.items():
            individual_cp = self.calculate_ideal_gas_cp(chem, self.temperature_K)
            if individual_cp is not None:
                mixture_cp += x * individual_cp
        return mixture_cp
    
    def _calculate_mixture_liquid_enthalpy(self):
        """
        Calculates the mixture's total liquid enthalpy based on a weighted average.
        
        Returns:
            float: The total enthalpy of the liquid mixture in J/mol.
        """
        mixture_h = 0.0
        for chem, x in self.mole_fractions.items():
            individual_h = self.calculate_liquid_enthalpy(chem, self.temperature_K)
            if individual_h is not None:
                mixture_h += x * individual_h
        return mixture_h

    def _calculate_mixture_gas_enthalpy(self):
        """
        Calculates the mixture's total gas enthalpy based on a weighted average.
        
        Returns:
            float: The total enthalpy of the gas mixture in J/mol.
        """
        mixture_h = 0.0
        for chem, x in self.mole_fractions.items():
            individual_h = self.calculate_ideal_enthalpy(chem, self.temperature_K)
            if individual_h is not None:
                mixture_h += x * individual_h
        return mixture_h
    
    def _calculate_mixture_heat_of_vaporization(self):
        """
        Calculates the mixture's molar heat of vaporization based on a weighted average.
        
        Returns:
            float: The mixture's heat of vaporization in J/mol.
        """
        mixture_h_vap = 0.0
        for chem, x in self.mole_fractions.items():
            individual_h_vap = self._calculate_molar_heat_of_vaporization(chem, self.temperature_K)
            if individual_h_vap is not None:
                mixture_h_vap += x * individual_h_vap
        return mixture_h_vap

    def _calculate_mixture_liquid_density(self):
        """
        Calculates the mixture's liquid density based on a weighted average of volumes.
        
        Returns:
            float: The mixture's liquid density in kg/m^3.
        """
        total_molar_volume = 0.0
        for chem, x in self.mole_fractions.items():
            individual_density = self.calculate_liquid_density(chem, self.temperature_K)
            if individual_density is not None:
                individual_molar_volume = (float(self._get_data(chem)['Mol_wt']) / 1000) / individual_density
                total_molar_volume += x * individual_molar_volume
        mixture_mol_wt = self.calculate_mixture_molar_weight()
        if total_molar_volume > 0:
            return (mixture_mol_wt / 1000) / total_molar_volume
        return None

    def _calculate_mixture_liquid_thermal_conductivity(self):
        """
        Calculates the mixture's liquid thermal conductivity based on a weighted average.
        
        Returns:
            float: The mixture's liquid thermal conductivity in W/(m*K).
        """
        mixture_k = 0.0
        for chem, x in self.mole_fractions.items():
            individual_k = self._calculate_liquid_thermal_conductivity(chem, self.temperature_K)
            if individual_k is not None:
                mixture_k += x * individual_k
        return mixture_k

    def _calculate_mixture_vapor_thermal_conductivity(self):
        """
        Calculates the mixture's vapor thermal conductivity based on a weighted average.
        
        Returns:
            float: The mixture's vapor thermal conductivity in W/(m*K).
        """
        mixture_k = 0.0
        for chem, x in self.mole_fractions.items():
            individual_k = self._calculate_vapor_thermal_conductivity(chem, self.temperature_K)
            if individual_k is not None:
                mixture_k += x * individual_k
        return mixture_k

    def calculate_ideal_molar_volume(self):
        """
        Calculates the ideal gas molar volume using the ideal gas law (V = RT/P).
        
        Returns:
            float: Molar volume in m^3/mol.
        """
        if self.pressure_Pa == 0:
            raise ValueError("Pressure cannot be zero for ideal gas law calculation.")
        return (self.R * self.temperature_K) / self.pressure_Pa

    def _calculate_bubble_point_temp(self):
        """
        Calculates the bubble point temperature using Raoult's Law and a bisection method.
        
        Returns:
            float: The bubble point temperature in Kelvin.
        """
        def bubble_point_function(T):
            sum_partial_pressures = 0
            for chem, x in self.mole_fractions.items():
                if x > 0:
                    P_sat = self.calculate_vapor_pressure(chem, T)
                    sum_partial_pressures += x * P_sat
            return self.pressure_Pa - sum_partial_pressures

        T_low = 100.0
        T_high = 1000.0
        tolerance = 1e-6
        max_iterations = 100

        for i in range(max_iterations):
            T_mid = (T_low + T_high) / 2
            f_mid = bubble_point_function(T_mid)

            if abs(f_mid) < tolerance:
                return T_mid
            
            f_low = bubble_point_function(T_low)
            if f_low * f_mid < 0:
                T_high = T_mid
            else:
                T_low = T_mid
                
        raise RuntimeError("Bisection method failed to converge for bubble point.")

    def _calculate_dew_point_temp(self):
        """
        Calculates the dew point temperature using Raoult's Law and a bisection method.
        
        Returns:
            float: The dew point temperature in Kelvin.
        """
        def dew_point_function(T):
            sum_of_inverse_pressures = 0
            for chem, y in self.mole_fractions.items():
                if y > 0:
                    P_sat = self.calculate_vapor_pressure(chem, T)
                    sum_of_inverse_pressures += y / P_sat
            return (1 / self.pressure_Pa) - sum_of_inverse_pressures
            
        T_low = 100.0
        T_high = 1000.0
        tolerance = 1e-6
        max_iterations = 100

        for i in range(max_iterations):
            T_mid = (T_low + T_high) / 2
            f_mid = dew_point_function(T_mid)

            if abs(f_mid) < tolerance:
                return T_mid
            
            f_low = dew_point_function(T_low)
            if f_low * f_mid < 0:
                T_high = T_mid
            else:
                T_low = T_mid
                
        raise RuntimeError("Bisection method failed to converge for dew point.")

    def _determine_phase(self):
        """
        Determines the phase of the mixture based on T, P, and composition.
        
        Returns:
            str: 'liquid', 'vapor', or 'two-phase'.
        """
        T_bubble = self._calculate_bubble_point_temp()
        T_dew = self._calculate_dew_point_temp()

        if self.temperature_K <= T_bubble:
            return 'liquid'
        elif self.temperature_K >= T_dew:
            return 'vapor'
        else:
            return 'two-phase'

    def _calculate_vapor_fraction(self):
        """
        Performs a simple flash calculation to determine the vapor fraction (beta)
        using the Rachford-Rice equation.
        
        Returns:
            float: The vapor fraction (beta) of the total flow.
        """
        def rachford_rice(beta, K_values, z):
            return sum([z[i] * (K_values[i] - 1) / (1 + beta * (K_values[i] - 1)) for i in range(len(K_values))])

        z = list(self.mole_fractions.values())
        K_values = [self.calculate_vapor_pressure(chem, self.temperature_K) / self.pressure_Pa for chem in self.chemicals]
        
        # Bisection method to solve for beta
        beta_low = 0.0
        beta_high = 1.0
        tolerance = 1e-6
        
        for _ in range(100):
            beta_mid = (beta_low + beta_high) / 2
            if beta_mid == 0 or beta_mid == 1:
                return beta_mid
            
            f_mid = rachford_rice(beta_mid, K_values, z)
            
            if abs(f_mid) < tolerance:
                return beta_mid
            
            f_low = rachford_rice(beta_low, K_values, z)
            if f_low * f_mid < 0:
                beta_high = beta_mid
            else:
                beta_low = beta_mid
                
        return beta_mid

    def calculate_all_mixture_properties(self):
        """
        Calculates all combined properties for the mixture based on its determined phase.
        
        Returns:
            dict: A dictionary of calculated mixture properties.
        """
        properties = {
            'Temperature (K)': self.temperature_K,
            'Pressure (Pa)': self.pressure_Pa,
            'Molar Flowrate (mol/s)': self.molar_flowrate_mol_s,
            'Mixture Molar Weight (g/mol)': self.calculate_mixture_molar_weight()
        }

        try:
            self.phase = self._determine_phase()
            properties['Phase'] = self.phase
        except RuntimeError as e:
            properties['Phase'] = f"Error determining phase: {e}"
            self.phase=None
            return properties

        if phase == 'liquid':
            self.mix_cp= self._calculate_mixture_liquid_cp()
            self.molar_enthalpy = self._calculate_mixture_liquid_enthalpy()
            self.total_enthalpy= self.molar_flowrate_mol_s * self.molar_enthalpy
            self.liquid_density= self._calculate_mixture_liquid_density()
            self.thermal_conductivity= self._calculate_mixture_liquid_thermal_conductivity()

            properties['Mixture Molar Cp (J/mol.K)'] = self.mix_cp
            properties['Mixture Molar Enthalpy (J/mol)'] = self.molar_enthalpy
            properties['Total Enthalpy Flow (W)'] = self.total_enthalpy
            properties['Mixture Liquid Density (kg/m^3)'] = self.liquid_density
            properties['Mixture Liquid Thermal Conductivity (W/m.K)'] = self.liqyid_thermal_conductivity

        elif phase == 'vapor':
            self.mix_cp = self._calculate_mixture_gas_cp()
            self.molar_enthalpy = self._calculate_mixture_gas_enthalpy()
            self.total_enthalpy = self.molar_flowrate_mol_s * self.molar_enthalpy
            self.thermal_conductivity= self._calculate_mixture_vapor_thermal_conductivity()


            properties['Mixture Molar Cp (J/mol.K)'] = self.mix_cp
            properties['Mixture Molar Enthalpy (J/mol)'] = self.molar_enthalpy
            properties['Total Enthalpy Flow (W)'] = self.total_enthalpy
            properties['Mixture Vapor Thermal Conductivity (W/m.K)'] = self.thermal_conductivity

        elif phase == 'two-phase':
            self.vapor_fraction = self._calculate_vapor_fraction()
            properties['Vapor Fraction'] = self.vapor_fraction
            
            liq_enthalpy = self._calculate_mixture_liquid_enthalpy()
            vap_enthalpy = self._calculate_mixture_gas_enthalpy()
            properties['Mixture Molar Enthalpy (J/mol)'] = (1 - vapor_fraction) * liq_enthalpy + vapor_fraction * vap_enthalpy
            properties['Total Enthalpy Flow (W)'] = self.molar_flowrate_mol_s * properties['Mixture Molar Enthalpy (J/mol)']
            
            liq_cp = self._calculate_mixture_liquid_cp()
            vap_cp = self._calculate_mixture_gas_cp()
            properties['Mixture Molar Cp (J/mol.K)'] = (1 - vapor_fraction) * liq_cp + vapor_fraction * vap_cp

            properties['Mixture Molar Heat of Vaporization (J/mol)'] = self._calculate_mixture_heat_of_vaporization()
            properties['Mixture Liquid Density (kg/m^3)'] = self._calculate_mixture_liquid_density()
            properties['Mixture Liquid Thermal Conductivity (W/m.K)'] = self._calculate_mixture_liquid_thermal_conductivity()
            properties['Mixture Vapor Thermal Conductivity (W/m.K)'] = self._calculate_mixture_vapor_thermal_conductivity()
            
        return properties

# --- Example Usage ---
if __name__ == "__main__":
    # Define the stream conditions and components with mole fractions
    chemicals_in_stream = {'Water': 0.8, 'Methane': 0.2}
    # This temperature should result in a two-phase mixture
    temperature = 200.0  # K
    pressure = 101325.0  # Pa (1 atm)
    molar_flowrate = 10.0 # mol/s
    
    # Create an instance of the property package
    stream_package = PropertyPackage(chemicals_in_stream, temperature, pressure, molar_flowrate)
    
    # Calculate all combined properties for the mixture
    mixture_properties = stream_package.calculate_all_mixture_properties()
    
    # Print the results
    print("--- Combined Mixture Properties Calculation ---")
    for key, value in mixture_properties.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")
    
    print("\n-------------------------------------")


TypeError: PropertyPackage.__init__() missing 1 required positional argument: 'molar_flowrate_mol_s'

In [ ]:
import csv
import io
import math
import pandas as pd

class PropertyPackage:
    """
    A class to calculate a wide range of properties for a stream of chemicals.

    This class loads chemical data from a database and provides methods to
    calculate individual and mixture properties for a given stream at a
    specific temperature and pressure, including phase determination.
    """
    
    # R: Ideal Gas Constant in J/(mol*K)
    R = 8.314
    
    def __init__(self, stream_name, chemicals_with_fractions, temperature_K, pressure_Pa, molar_flowrate_mol_s):
        """
        Initializes the property package with a dictionary of chemicals, T, P, and flow rate.

        Args:
            chemicals_with_fractions (dict): A dictionary of chemical names and their
                                             mole fractions (e.g., {'Methane': 0.8, 'Water': 0.2}).
            temperature_K (float): The stream temperature in Kelvin.
            pressure_Pa (float): The stream pressure in Pascals.
            molar_flowrate_mol_s (float): The total molar flow rate of the stream in mol/s.
        """
        # Store the chemical names and mole fractions
        self.stream_name = stream_name
        self.chemicals = list(chemicals_with_fractions.keys())
        self.mole_fractions = chemicals_with_fractions
        self.temperature_K = temperature_K
        self.pressure_Pa = pressure_Pa
        self.molar_flowrate_mol_s = molar_flowrate_mol_s
        
        # Load data from the database using pandas as requested
        # Note: You'll need an 'Open_soruce_db.csv' file in the same directory
        # In a real application, you might use a more robust database connection
        try:
            self.data = pd.read_csv('Open_soruce_db.csv')
        except FileNotFoundError:
            print("Warning: 'Open_soruce_db.csv' not found. Using a simulated database.")
            # Use the in-memory data if the file is not found
            self.data = self._load_simulated_database()
            
        # Initialize phase-specific properties to None
        self.phase = None
        self.vapor_fraction = None
        self.liquid_mole_fractions = None
        self.vapor_mole_fractions = None

    def _load_simulated_database(self):
        """
        Loads the simulated chemical data from the internal CSV string into a pandas DataFrame.
        This is a fallback if the external CSV is not present.
        """
        CHEMICAL_DATABASE_CSV = """
Name,Critical_Temperature,Critical_Temp_unit,Critical_Pressure,Critical_pressure_unit,Critical_Volume,Critical_vol_unit,Critical_Compressibility,Boiling_point,Boiling_point_unit,Mol_wt,Liquid_volume_at_BP,Liquid_vol_unit_at_BP,AcentricityFactor,Heat_of_Formation,Heat_of_Formation_unit,Gibbs_Energy_Of_Formation,Gibbs_Energy_Of_Formation_unit,Heat_of_combustion,Heat_of_combustion_unit,Lq_Den_A,Lq_Den_B,Lq_Den_C,Lq_Den_D,Lq_Den_Tmin,Lq_Den_Tmax,Lq_Den_unit,Vap_P_A,Vap_P_B,Vap_P_C,Vap_P_D,Vap_P_E,Vap_P_Tmin,Vap_P_Tmax,Vap_P_unit,Liq_Cp_A,Liq_Cp_B,Liq_Cp_C,Liq_Cp_D,Liq_Cp_E,Liq_Cp_Tmin,Liq_Cp_Tmax,Liq_Cp_unit,Gas_Cp_A,Gas_Cp_B,Gas_Cp_C,Gas_Cp_D,Gas_Cp_E,Gas_Cp_Tmin,Gas_Cp_Tmax,Gas_Cp_unit,Liq_vis_A,Liq_vis_B,Liq_vis_C,Liq_vis_D,Liq_vis_E,Liq_vis_Tmin,Liq_vis_Tmax,Liq_vis_unit,Liq_Therm_con_A,Liq_Therm_con_B,Liq_Therm_con_C,Liq_Therm_con_D,Liq_Therm_con_E,Liq_Therm_con_Tmin,Liq_Therm_con_Tmax,Liq_Therm_con_unit,Vap_Therm_con_A,Vap_Therm_con_B,Vap_Therm_con_C,Vap_Therm_con_D,Vap_Therm_con_Tmin,Vap_Therm_con_Tmax,Vap_Therm_con_unit,Heat_of_vap_A,Heat_of_vap_B,Heat_of_vap_C,Heat_of_vap_D,Heat_of_vap_E,Heat_of_vap_Tmin,Heat_of_vap_Tmax,Heat_of_vap_unit
Methane,190.56,K,4.599,MPa,,,,,111.66,K,16.042,,,-0.011,-74.85,kJ/mol,,,,,145.42,1.385,258.96,,111.7,190.56,kg/m3,6.61113,-641.527,1.2584e-09,2,0.0,,,,,5.539,0.0305,0.0001,,298.15,1000,J/(mol K),34.331,-5.465e-02,3.159e-04,-1.491e-07,,298.15,1500,J/(mol K),,,,,,,,,,,0.005574,0.01633,0.000032,0.0,,,,0.00396,0.000045,0.0000000003,,200,600,W/m.K,242.0,-2.36,0.0,0.0,0.0,111.66,190.56,J/mol
Water,647.1,K,22.06,MPa,,,,,373.15,K,18.015,,,-0.344,-285.83,kJ/mol,,,,,999.0,0.000,273.15,373.15,kg/m3,8.07131,1730.63,233.426,0.0,0.0,,,,,61.43,3.023E-02,0.0,0.0,0.0,273.15,373.15,J/(mol K),33.32,-0.2526e-02,-0.201e-05,0.002e-08,,298.15,1000,J/(mol K),33.32,-0.2526e-02,-0.201e-05,0.002e-08,,298.15,1000,J/(mol K),0.0005,0.000033,-0.0000000003,,273.15,647.1,W/m.K,45000.0,-50.0,0.0,0.0,0.0,273.15,647.1,J/mol
"""
        return pd.read_csv(io.StringIO(CHEMICAL_DATABASE_CSV))

    def _get_data(self, chemical_name):
        """
        Helper method to retrieve data for a specific chemical from the DataFrame.
        Returns a pandas Series for the specified chemical's row.
        """
        try:
            return self.data.loc[self.data['Name'] == chemical_name].iloc[0]
        except IndexError:
            raise ValueError(f"Chemical '{chemical_name}' not found in database.")

    # --- Individual Property Calculation Methods ---

    def _calculate_ideal_gas_cp(self, chemical_name, temperature_K):
        """
        Calculates the ideal gas molar heat capacity (Cp) for a chemical using a polynomial.
        Formula: Cp = A + B*T + C*T^2 + D*T^3 + E*T^4
        
        Returns:
            float: Molar heat capacity in J/(mol*K).
        """
        chem_data = self._get_data(chemical_name)
        T = temperature_K
        
        try:
            A = float(chem_data['Gas_Cp_A'])
            B = float(chem_data['Gas_Cp_B'])
            C = float(chem_data['Gas_Cp_C'])
            D = float(chem_data['Gas_Cp_D'])
            E = float(chem_data.get('Gas_Cp_E', 0.0))
            
            cp = A + B * T + C * (T**2) + D * (T**3) + E * (T**4)
            return cp
        except (ValueError, KeyError) as e:
            return None

    def _calculate_ideal_enthalpy(self, chemical_name, temperature_K, T_ref=298.15):
        """
        Calculates the ideal gas molar enthalpy (H) for a chemical relative to a
        reference temperature (T_ref) and the heat of formation.
        
        Returns:
            float: Molar enthalpy in J/mol.
        """
        chem_data = self._get_data(chemical_name)
        T = temperature_K
        
        try:
            H_form = float(chem_data['Heat_of_Formation']) * 1000
            A = float(chem_data['Gas_Cp_A'])
            B = float(chem_data['Gas_Cp_B'])
            C = float(chem_data['Gas_Cp_C'])
            D = float(chem_data['Gas_Cp_D'])
            E = float(chem_data.get('Gas_Cp_E', 0.0))
            
            sensible_heat = (
                A * (T - T_ref) +
                (B/2) * (T**2 - T_ref**2) +
                (C/3) * (T**3 - T_ref**3) +
                (D/4) * (T**4 - T_ref**4) +
                (E/5) * (T**5 - T_ref**5)
            )
            
            total_enthalpy = H_form + sensible_heat
            return total_enthalpy
            
        except (ValueError, KeyError) as e:
            return None

    def _calculate_liquid_enthalpy(self, chemical_name, temperature_K, T_ref=298.15):
        """
        Calculates the liquid molar enthalpy (H) for a chemical relative to a
        reference temperature (T_ref) and the heat of formation.
        
        Returns:
            float: Molar enthalpy in J/mol.
        """
        chem_data = self._get_data(chemical_name)
        T = temperature_K
        
        try:
            H_form = float(chem_data['Heat_of_Formation']) * 1000
            A = float(chem_data['Liq_Cp_A'])
            B = float(chem_data['Liq_Cp_B'])
            C = float(chem_data['Liq_Cp_C'])
            D = float(chem_data.get('Liq_Cp_D', 0.0))
            E = float(chem_data.get('Liq_Cp_E', 0.0))
            
            sensible_heat = (
                A * (T - T_ref) +
                (B/2) * (T**2 - T_ref**2) +
                (C/3) * (T**3 - T_ref**3) +
                (D/4) * (T**4 - T_ref**4) +
                (E/5) * (T**5 - T_ref**5)
            )
            
            total_enthalpy = H_form + sensible_heat
            return total_enthalpy
            
        except (ValueError, KeyError) as e:
            return None

    def _calculate_vapor_pressure(self, chemical_name, temperature_K):
        """
        Calculates the vapor pressure (P_sat) for a chemical using an Antoine-like equation.
        
        Returns:
            float: Vapor pressure in Pascals.
        """
        chem_data = self._get_data(chemical_name)
        
        try:
            A = float(chem_data['Vap_P_A'])
            B = float(chem_data['Vap_P_B'])
            C = float(chem_data.get('Vap_P_C', 0.0))
            D = float(chem_data.get('Vap_P_D', 0.0))
            E = float(chem_data.get('Vap_P_E', 1.0))
            
            log_P_sat = A + B/temperature_K + C * math.log(temperature_K) + D * (temperature_K**E)
            P_sat_Pa = math.exp(log_P_sat)
            return P_sat_Pa
            
        except (ValueError, KeyError) as e:
            raise ValueError(f"Error calculating vapor pressure for {chemical_name}: {e}")

    def _calculate_liquid_density(self, chemical_name, temperature_K):
        """
        Calculates the liquid density (rho) for a chemical using a polynomial.
        Formula: rho = A + B*T + C*T^2 + D*T^3
        
        Returns:
            float: Liquid density in kg/m^3.
        """
        chem_data = self._get_data(chemical_name)
        T = temperature_K
        
        try:
            A = float(chem_data['Lq_Den_A'])
            B = float(chem_data['Lq_Den_B'])
            C = float(chem_data.get('Lq_Den_C', 0.0))
            D = float(chem_data.get('Lq_Den_D', 0.0))
            
            density = A + B * T + C * (T**2) + D * (T**3)
            return density
        except (ValueError, KeyError) as e:
            return None
            
    def _calculate_vapor_density(self, chemical_name, temperature_K, pressure_Pa):
        """
        Calculates the vapor density (rho) for a chemical using ideal gas law.
        Formula: rho = P*MW/(R*T)
        
        Returns:
            float: Vapor density in kg/m^3.
        """
        chem_data = self._get_data(chemical_name)
        T = temperature_K
        P = pressure_Pa
        
        try:
            mol_wt = float(chem_data['Mol_wt'])
            density = (P * (mol_wt / 1000)) / (self.R * T)
            return density
        except (ValueError, KeyError) as e:
            return None
            
    def _calculate_liquid_cp(self, chemical_name, temperature_K):
        """
        Calculates the liquid molar heat capacity (Cp) for a chemical using a polynomial.
        
        Returns:
            float: Liquid molar heat capacity in J/(mol*K).
        """
        chem_data = self._get_data(chemical_name)
        T = temperature_K
        
        try:
            A = float(chem_data['Liq_Cp_A'])
            B = float(chem_data['Liq_Cp_B'])
            C = float(chem_data['Liq_Cp_C'])
            D = float(chem_data.get('Liq_Cp_D', 0.0))
            E = float(chem_data.get('Liq_Cp_E', 0.0))
            
            cp = A + B * T + C * (T**2) + D * (T**3) + E * (T**4)
            return cp
        except (ValueError, KeyError) as e:
            return None

    def _calculate_molar_heat_of_vaporization(self, chemical_name, temperature_K):
        """
        Calculates the molar heat of vaporization (H_vap) for a chemical using a polynomial.
        
        Returns:
            float: Molar heat of vaporization in J/mol.
        """
        chem_data = self._get_data(chemical_name)
        T = temperature_K
        
        try:
            A = float(chem_data['Heat_of_vap_A'])
            B = float(chem_data['Heat_of_vap_B'])
            C = float(chem_data.get('Heat_of_vap_C', 0.0))
            D = float(chem_data.get('Heat_of_vap_D', 0.0))
            E = float(chem_data.get('Heat_of_vap_E', 0.0))
            
            h_vap = A + B * T + C * (T**2) + D * (T**3) + E * (T**4)
            return h_vap
        except (ValueError, KeyError) as e:
            return None

    def _calculate_liquid_thermal_conductivity(self, chemical_name, temperature_K):
        """
        Calculates the liquid thermal conductivity (k) for a chemical using a polynomial.
        Formula: k = A + B*T + C*T^2 + D*T^3 + E*T^4
        
        Returns:
            float: Thermal conductivity in W/(m*K).
        """
        chem_data = self._get_data(chemical_name)
        T = temperature_K
        
        try:
            A = float(chem_data['Liq_Therm_con_A'])
            B = float(chem_data['Liq_Therm_con_B'])
            C = float(chem_data['Liq_Therm_con_C'])
            D = float(chem_data.get('Liq_Therm_con_D', 0.0))
            E = float(chem_data.get('Liq_Therm_con_E', 0.0))
            
            k_liq = A + B * T + C * (T**2) + D * (T**3) + E * (T**4)
            return k_liq
        except (ValueError, KeyError) as e:
            return None

    def _calculate_vapor_thermal_conductivity(self, chemical_name, temperature_K):
        """
        Calculates the vapor thermal conductivity (k) for a chemical using a polynomial.
        Formula: k = A + B*T + C*T^2 + D*T^3
        
        Returns:
            float: Thermal conductivity in W/(m*K).
        """
        chem_data = self._get_data(chemical_name)
        T = temperature_K
        
        try:
            A = float(chem_data['Vap_Therm_con_A'])
            B = float(chem_data['Vap_Therm_con_B'])
            C = float(chem_data['Vap_Therm_con_C'])
            D = float(chem_data.get('Vap_Therm_con_D', 0.0))
            
            k_vap = A + B * T + C * (T**2) + D * (T**3)
            return k_vap
        except (ValueError, KeyError) as e:
            return None
    
    # --- Mixture Property Calculation Methods ---

    def _calculate_mixture_molar_weight(self, mole_fractions):
        """
        Calculates the mixture's molar weight based on a weighted average of components.
        
        Returns:
            float: The molar weight of the mixture in g/mol.
        """
        mixture_mw = 0.0
        for chem, x in mole_fractions.items():
            chem_data = self._get_data(chem)
            mw = float(chem_data['Mol_wt'])
            mixture_mw += x * mw
        return mixture_mw

    def _calculate_mixture_liquid_cp(self, mole_fractions):
        """
        Calculates the mixture's liquid heat capacity based on a weighted average.
        
        Returns:
            float: The mixture's liquid heat capacity in J/(mol*K).
        """
        mixture_cp = 0.0
        for chem, x in mole_fractions.items():
            individual_cp = self._calculate_liquid_cp(chem, self.temperature_K)
            if individual_cp is not None:
                mixture_cp += x * individual_cp
        return mixture_cp

    def _calculate_mixture_gas_cp(self, mole_fractions):
        """
        Calculates the mixture's ideal gas heat capacity based on a weighted average.
        
        Returns:
            float: The mixture's ideal gas heat capacity in J/(mol*K).
        """
        mixture_cp = 0.0
        for chem, x in mole_fractions.items():
            individual_cp = self._calculate_ideal_gas_cp(chem, self.temperature_K)
            if individual_cp is not None:
                mixture_cp += x * individual_cp
        return mixture_cp
    
    def _calculate_mixture_liquid_enthalpy(self, mole_fractions):
        """
        Calculates the mixture's total liquid enthalpy based on a weighted average.
        
        Returns:
            float: The total enthalpy of the liquid mixture in J/mol.
        """
        mixture_h = 0.0
        for chem, x in mole_fractions.items():
            individual_h = self._calculate_liquid_enthalpy(chem, self.temperature_K)
            if individual_h is not None:
                mixture_h += x * individual_h
        return mixture_h

    def _calculate_mixture_gas_enthalpy(self, mole_fractions):
        """
        Calculates the mixture's total gas enthalpy based on a weighted average.
        
        Returns:
            float: The total enthalpy of the gas mixture in J/mol.
        """
        mixture_h = 0.0
        for chem, x in mole_fractions.items():
            individual_h = self._calculate_ideal_enthalpy(chem, self.temperature_K)
            if individual_h is not None:
                mixture_h += x * individual_h
        return mixture_h
    
    def _calculate_mixture_heat_of_vaporization(self, mole_fractions):
        """
        Calculates the mixture's molar heat of vaporization based on a weighted average.
        
        Returns:
            float: The mixture's heat of vaporization in J/mol.
        """
        mixture_h_vap = 0.0
        for chem, x in mole_fractions.items():
            individual_h_vap = self._calculate_molar_heat_of_vaporization(chem, self.temperature_K)
            if individual_h_vap is not None:
                mixture_h_vap += x * individual_h_vap
        return mixture_h_vap

    def _calculate_mixture_liquid_density(self, mole_fractions):
        """
        Calculates the mixture's liquid density based on a weighted average of volumes.
        
        Returns:
            float: The mixture's liquid density in kg/m^3.
        """
        total_molar_volume = 0.0
        mixture_mol_wt = self._calculate_mixture_molar_weight(mole_fractions)
        for chem, x in mole_fractions.items():
            individual_density = self._calculate_liquid_density(chem, self.temperature_K)
            if individual_density is not None and individual_density > 0:
                individual_molar_volume = (float(self._get_data(chem)['Mol_wt']) / 1000) / individual_density
                total_molar_volume += x * individual_molar_volume
        
        if total_molar_volume > 0:
            return (mixture_mol_wt / 1000) / total_molar_volume
        return None

    def _calculate_mixture_liquid_thermal_conductivity(self, mole_fractions):
        """
        Calculates the mixture's liquid thermal conductivity based on a weighted average.
        
        Returns:
            float: The mixture's liquid thermal conductivity in W/(m*K).
        """
        mixture_k = 0.0
        for chem, x in mole_fractions.items():
            individual_k = self._calculate_liquid_thermal_conductivity(chem, self.temperature_K)
            if individual_k is not None:
                mixture_k += x * individual_k
        return mixture_k

    def _calculate_mixture_vapor_thermal_conductivity(self, mole_fractions):
        """
        Calculates the mixture's vapor thermal conductivity based on a weighted average.
        
        Returns:
            float: The mixture's vapor thermal conductivity in W/(m*K).
        """
        mixture_k = 0.0
        for chem, x in mole_fractions.items():
            individual_k = self._calculate_vapor_thermal_conductivity(chem, self.temperature_K)
            if individual_k is not None:
                mixture_k += x * individual_k
        return mixture_k

    def calculate_ideal_molar_volume(self):
        """
        Calculates the ideal gas molar volume using the ideal gas law (V = RT/P).
        
        Returns:
            float: Molar volume in m^3/mol.
        """
        if self.pressure_Pa == 0:
            raise ValueError("Pressure cannot be zero for ideal gas law calculation.")
        return (self.R * self.temperature_K) / self.pressure_Pa

    def _calculate_bubble_point_temp(self):
        """
        Calculates the bubble point temperature using Raoult's Law and a bisection method.
        
        Returns:
            float: The bubble point temperature in Kelvin.
        """
        def bubble_point_function(T):
            sum_partial_pressures = 0
            for chem, x in self.mole_fractions.items():
                if x > 0:
                    P_sat = self._calculate_vapor_pressure(chem, T)
                    sum_partial_pressures += x * P_sat
            return self.pressure_Pa - sum_partial_pressures

        T_low = 100.0
        T_high = 1000.0
        tolerance = 1e-6
        max_iterations = 100

        for i in range(max_iterations):
            T_mid = (T_low + T_high) / 2
            f_mid = bubble_point_function(T_mid)

            if abs(f_mid) < tolerance:
                return T_mid
            
            f_low = bubble_point_function(T_low)
            if f_low * f_mid < 0:
                T_high = T_mid
            else:
                T_low = T_mid
                
        raise RuntimeError("Bisection method failed to converge for bubble point.")

    def _calculate_dew_point_temp(self):
        """
        Calculates the dew point temperature using Raoult's Law and a bisection method.
        
        Returns:
            float: The dew point temperature in Kelvin.
        """
        def dew_point_function(T):
            sum_of_inverse_pressures = 0
            for chem, y in self.mole_fractions.items():
                if y > 0:
                    P_sat = self._calculate_vapor_pressure(chem, T)
                    sum_of_inverse_pressures += y / P_sat
            return (1 / self.pressure_Pa) - sum_of_inverse_pressures
            
        T_low = 100.0
        T_high = 1000.0
        tolerance = 1e-6
        max_iterations = 100

        for i in range(max_iterations):
            T_mid = (T_low + T_high) / 2
            f_mid = dew_point_function(T_mid)

            if abs(f_mid) < tolerance:
                return T_mid
            
            f_low = dew_point_function(T_low)
            if f_low * f_mid < 0:
                T_high = T_mid
            else:
                T_low = T_mid
                
        raise RuntimeError("Bisection method failed to converge for dew point.")

    def _determine_phase(self):
        """
        Determines the phase of the mixture based on T, P, and composition.
        
        Returns:
            str: 'liquid', 'vapor', or 'two-phase'.
        """
        try:
            T_bubble = self._calculate_bubble_point_temp()
            T_dew = self._calculate_dew_point_temp()
        except Exception as e:
            print(f"Error in phase determination: {e}")
            return "Error"

        if self.temperature_K <= T_bubble:
            return 'liquid'
        elif self.temperature_K >= T_dew:
            return 'vapor'
        else:
            return 'two-phase'

    def _calculate_vapor_fraction(self):
        """
        Performs a simple flash calculation to determine the vapor fraction (beta)
        using the Rachford-Rice equation.
        
        Returns:
            float: The vapor fraction (beta) of the total flow.
        """
        def rachford_rice(beta, K_values, z):
            return sum([z[i] * (K_values[i] - 1) / (1 + beta * (K_values[i] - 1)) for i in range(len(K_values))])

        z = list(self.mole_fractions.values())
        K_values = [self._calculate_vapor_pressure(chem, self.temperature_K) / self.pressure_Pa for chem in self.chemicals]
        
        # Bisection method to solve for beta
        beta_low = 0.0
        beta_high = 1.0
        tolerance = 1e-6
        
        for _ in range(100):
            beta_mid = (beta_low + beta_high) / 2
            if beta_mid == 0 or beta_mid == 1:
                return beta_mid
            
            f_mid = rachford_rice(beta_mid, K_values, z)
            
            if abs(f_mid) < tolerance:
                return beta_mid
            
            f_low = rachford_rice(beta_low, K_values, z)
            if f_low * f_mid < 0:
                beta_high = beta_mid
            else:
                beta_low = beta_mid
                
        return beta_mid

    def _calculate_phase_compositions(self):
        """
        Calculates the mole fractions of the liquid and vapor phases in a two-phase mixture.
        
        Returns:
            tuple: A tuple containing two dictionaries, (liquid_mole_fractions, vapor_mole_fractions).
        """
        if self.phase != 'two-phase' or self.vapor_fraction is None:
            return {}, {}
            
        liquid_mole_fractions = {}
        vapor_mole_fractions = {}
        
        K_values = {chem: self._calculate_vapor_pressure(chem, self.temperature_K) / self.pressure_Pa for chem in self.chemicals}
        
        for chem in self.chemicals:
            z_i = self.mole_fractions[chem]
            K_i = K_values[chem]
            beta = self.vapor_fraction
            
            # Rachford-Rice derived equations for phase compositions
            x_i = z_i / (1 + beta * (K_i - 1))
            y_i = z_i * K_i / (1 + beta * (K_i - 1))
            
            liquid_mole_fractions[chem] = x_i
            vapor_mole_fractions[chem] = y_i

        # Normalize the mole fractions to ensure they sum to 1
        sum_x = sum(liquid_mole_fractions.values())
        sum_y = sum(vapor_mole_fractions.values())
        
        liquid_mole_fractions = {k: v / sum_x for k, v in liquid_mole_fractions.items()}
        vapor_mole_fractions = {k: v / sum_y for k, v in vapor_mole_fractions.items()}
        
        return liquid_mole_fractions, vapor_mole_fractions


    def calculate_all_mixture_properties(self):
        """
        Calculates all combined properties for the mixture based on its determined phase.
        
        Returns:
            dict: A dictionary of calculated mixture properties.
        """
        # Overall stream properties (independent of phase)
        
        properties = {
            'Temperature (K)': self.temperature_K,
            'Pressure (Pa)': self.pressure_Pa,
            'Molar Flowrate (mol/s)': self.molar_flowrate_mol_s,
            'Mixture Molar Weight (g/mol)': self._calculate_mixture_molar_weight(self.mole_fractions)
        }

        # Determine the phase
        self.phase = self._determine_phase()
        properties['Phase'] = self.phase

        if self.phase == 'liquid':
            properties['Mixture Molar Cp (J/mol.K)'] = self._calculate_mixture_liquid_cp(self.mole_fractions)
            properties['Mixture Molar Enthalpy (J/mol)'] = self._calculate_mixture_liquid_enthalpy(self.mole_fractions)
            properties['Total Enthalpy Flow (W)'] = self.molar_flowrate_mol_s * properties['Mixture Molar Enthalpy (J/mol)']
            properties['Mixture Density (kg/m^3)'] = self._calculate_mixture_liquid_density(self.mole_fractions)
            properties['Mixture Thermal Conductivity (W/m.K)'] = self._calculate_mixture_liquid_thermal_conductivity(self.mole_fractions)

        elif self.phase == 'vapor':
            properties['Mixture Molar Cp (J/mol.K)'] = self._calculate_mixture_gas_cp(self.mole_fractions)
            properties['Mixture Molar Enthalpy (J/mol)'] = self._calculate_mixture_gas_enthalpy(self.mole_fractions)
            properties['Total Enthalpy Flow (W)'] = self.molar_flowrate_mol_s * properties['Mixture Molar Enthalpy (J/mol)']
            properties['Mixture Thermal Conductivity (W/m.K)'] = self._calculate_mixture_vapor_thermal_conductivity(self.mole_fractions)
            
        elif self.phase == 'two-phase':
            self.vapor_fraction = self._calculate_vapor_fraction()
            properties['Vapor Fraction'] = self.vapor_fraction

            # Calculate phase-specific compositions
            self.liquid_mole_fractions, self.vapor_mole_fractions = self._calculate_phase_compositions()
            
            # Liquid Phase Properties
            liq_enthalpy = self._calculate_mixture_liquid_enthalpy(self.liquid_mole_fractions)
            liq_cp = self._calculate_mixture_liquid_cp(self.liquid_mole_fractions)
            liq_density = self._calculate_mixture_liquid_density(self.liquid_mole_fractions)
            liq_thermal_k = self._calculate_mixture_liquid_thermal_conductivity(self.liquid_mole_fractions)
            
            # Vapor Phase Properties
            vap_enthalpy = self._calculate_mixture_gas_enthalpy(self.vapor_mole_fractions)
            vap_cp = self._calculate_mixture_gas_cp(self.vapor_mole_fractions)
            vap_thermal_k = self._calculate_mixture_vapor_thermal_conductivity(self.vapor_mole_fractions)
            
            # Combined Stream Properties (weighted by vapor fraction)
            properties['Mixture Molar Enthalpy (J/mol)'] = (1 - self.vapor_fraction) * liq_enthalpy + self.vapor_fraction * vap_enthalpy
            properties['Total Enthalpy Flow (W)'] = self.molar_flowrate_mol_s * properties['Mixture Molar Enthalpy (J/mol)']
            properties['Mixture Molar Cp (J/mol.K)'] = (1 - self.vapor_fraction) * liq_cp + self.vapor_fraction * vap_cp
            properties['Mixture Molar Heat of Vaporization (J/mol)'] = self._calculate_mixture_heat_of_vaporization(self.mole_fractions)
            
            # Add phase-specific compositions and properties to the output dictionary
            properties['Liquid Phase Mole Fractions'] = self.liquid_mole_fractions
            properties['Liquid Phase Molar Cp (J/mol.K)'] = liq_cp
            properties['Liquid Phase Molar Enthalpy (J/mol)'] = liq_enthalpy
            properties['Liquid Phase Density (kg/m^3)'] = liq_density
            properties['Liquid Phase Thermal Conductivity (W/m.K)'] = liq_thermal_k

            properties['Vapor Phase Mole Fractions'] = self.vapor_mole_fractions
            properties['Vapor Phase Molar Cp (J/mol.K)'] = vap_cp
            properties['Vapor Phase Molar Enthalpy (J/mol)'] = vap_enthalpy
            properties['Vapor Phase Thermal Conductivity (W/m.K)'] = vap_thermal_k

        return properties

# --- Example Usage ---
if __name__ == "__main__":
    # Define the stream conditions and components with mole fractions
    # This temperature should result in a two-phase mixture
    chemicals_in_stream = {'Water': 0.8, 'Methane': 0.2}
    temperature = 200.0   # K
    pressure = 101325.0   # Pa (1 atm)
    molar_flowrate = 10.0 # mol/s
    
    # Create an instance of the property package
    stream_package = PropertyPackage('Two-Phase Stream', chemicals_in_stream, temperature, pressure, molar_flowrate)
    
    # Calculate all combined properties for the mixture
    mixture_properties = stream_package.calculate_all_mixture_properties()
    
    # Print the results
    print("--- Combined Mixture Properties Calculation ---")
    for key, value in mixture_properties.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        elif isinstance(value, dict):
            print(f"  {key}:")
            for sub_key, sub_value in value.items():
                print(f"    - {sub_key}: {sub_value:.4f}")
        else:
            print(f"  {key}: {value}")
    
    print("\n-------------------------------------")


--- Combined Mixture Properties Calculation ---
  Temperature (K): 200.0000
  Pressure (Pa): 101325.0000
  Molar Flowrate (mol/s): 10.0000
  Mixture Molar Weight (g/mol): 17.6207
  Phase: two-phase
  Vapor Fraction: 0.1864
  Mixture Molar Enthalpy (J/mol): -208430917060.0944
  Total Enthalpy Flow (W): -2084309170600.9443
  Mixture Molar Cp (J/mol.K): 461839.7427
  Mixture Molar Heat of Vaporization (J/mol): 0.0000
  Liquid Phase Mole Fractions:
    - Water: 0.9833
    - Methane: 0.0167
  Liquid Phase Molar Cp (J/mol.K): 562889.0000
  Liquid Phase Molar Enthalpy (J/mol): -239110494259.5909
  Liquid Phase Density (kg/m^3): 630414826.7989
  Liquid Phase Thermal Conductivity (W/m.K): -51414.6220
  Vapor Phase Mole Fractions:
    - Water: 0.0000
    - Methane: 1.0000
  Vapor Phase Molar Cp (J/mol.K): 20786.0000
  Vapor Phase Molar Enthalpy (J/mol): -74522537105.1314
  Vapor Phase Thermal Conductivity (W/m.K): 111939106963.8461

-------------------------------------


In [1]:
my_dict = {'a': 10, 'b': 20, 'c': 30}
total = sum(my_dict.values())
print("Sum of all values:", total)

Sum of all values: 60


In [4]:
original_dict = {'a': 10, 'b': 20, 'c': 30}
total_sum = sum(original_dict.values())

# Create a new dictionary with each value divided by the total sum
normalized_dict = {key: value / total_sum for key, value in original_dict.items()}

print(normalized_dict)

{'a': 0.16666666666666666, 'b': 0.3333333333333333, 'c': 0.5}


In [3]:
original_dict.values()

dict_values([10, 20, 30])